# Overview
Model 2 is a **Hybrid Multi-Layer Perceptron (MLP) ranking model** that is designed to rank news articles for a user to browse through. It combines text data (such as BERT embeddings) with user behavior and reading profile to decide which articles a user is most likely to click. We use a listwise ranking approach and the model learns by comparing groups of articles together to figure out the best order for a recommendation list.

The document is structured as follows:

* **Sections 1-6: Baseline Implementation** - Outlines the construction of the initial model iteration, incorporating **all** engineered features, K-Means clusters, and BERT embeddings into the MLP architecture.
* **Sections 7–9: Performance & Interpretability Analysis** -
Evaluates model performance through metric tracking and Feature Attribution (Integrated Gradients). This phase identifies signal dominance, validates feature utility, and checks for architectural biases.
* **Section 10: Final Iteration** - We consolidate our findings to produce our finalised version of the MLP model.

#1 Load Datasets

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load NEWS datasets
news_ent_path = '/content/drive/MyDrive/data/news/news_entities.parquet'
news_features_path = '/content/drive/MyDrive/features/news_features/news_features.parquet'
news_entities = pd.read_parquet(news_ent_path)
news_features = pd.read_parquet(news_features_path)

In [ ]:
# Load USER datasets
user_path = '/content/drive/MyDrive/features/user_features'
train_user_features = pd.read_parquet(f'{user_path}/train_user_features.parquet')
val_user_features = pd.read_parquet(f'{user_path}/val_user_features.parquet')
test_user_features = pd.read_parquet(f'{user_path}/test_user_features.parquet')

In [ ]:
print(len(train_user_features))
print(len(val_user_features))
print(len(test_user_features))

141558
31624
30270


In [ ]:
# Load CONTEXT datasets
context_path = '/content/drive/MyDrive/features/context_features'
train_context_features = pd.read_parquet(f'{context_path}/train_context.parquet')
val_context_features = pd.read_parquet(f'{context_path}/val_context.parquet')
test_context_features = pd.read_parquet(f'{context_path}/test_context.parquet')

In [ ]:
print(len(train_context_features))
print(len(val_context_features))
print(len(test_context_features))

141558
31624
30270


In [ ]:
# Load INTERACTION dataset
train_interaction_path = '/content/drive/MyDrive/features/interaction_features/train_interactions.parquet'
train_interactions = pd.read_parquet(train_interaction_path)
val_interaction_path = '/content/drive/MyDrive/features/interaction_features/val_interactions.parquet'
val_interactions = pd.read_parquet(val_interaction_path)
test_interaction_path = '/content/drive/MyDrive/features/interaction_features/test_interactions.parquet'
test_interactions = pd.read_parquet(test_interaction_path)

In [ ]:
print(len(train_interactions))
print(len(val_interactions))
print(len(test_interactions))

141558
31624
30270


In [ ]:
# Load entity embedding
entity_emb_path = '/content/drive/MyDrive/embeddings/news_entities.parquet'
entity_emb = pd.read_parquet(entity_emb_path)

In [ ]:
# Extract the BERT embeddings

bert_embeddings = news_features.filter(regex='bert_emb_').to_numpy(dtype=np.float32)
news_features = news_features.drop(columns=news_features.filter(regex='bert_emb_').columns)

In [ ]:
# Extract latent topic embeddings from news_features

latent_topics = news_features.filter(regex='topic_').to_numpy(dtype=np.float32)
news_features = news_features.drop(columns=news_features.filter(regex='topic_').columns)

In [ ]:
entity_emb.head()

,news_id,kg_title_emb,kg_abstract_emb,kg_title_flag,kg_abstract_flag,ner_title_emb,ner_abstract_emb,ner_title_flag,ner_abstract_flag
0,N55528,"[0.008122858, -0.07991525, -0.016764905, 0.158...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,0,"[-0.025332777, 0.07448415, 0.046060078, 0.0411...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,0
1,N19639,"[-0.029497119, -0.021168852, 0.03713986, -0.11...","[-0.029497119, -0.021168852, 0.03713986, -0.11...",1,1,"[-0.007022121, 0.08226046, -0.035310447, 0.089...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,0
2,N61837,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.06553931, -0.08845358, -0.015253108, -0.03...",0,1,"[-0.07378371, 0.10624181, 0.00063977536, 0.055...","[0.016665561, 0.026591385, -0.07544741, 0.0324...",1,1
3,N53526,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0061505293, -0.10125916, -0.060772542, 0.04...",0,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.008782822, 0.051812273, -0.033509996, -0.0...",0,1
4,N38324,"[0.045086294, 0.05803315, 0.016441302, 0.00560...","[-0.02720732, -0.0005318837, 0.08588548, -0.13...",1,1,"[-0.0391243, 0.029080106, 0.014465057, 0.06514...","[-0.0037145745, -0.02635876, -0.027976032, 0.0...",1,1


In [ ]:
print(bert_embeddings[:5])

[[-0.00656653  0.03001126  0.041711   ... -0.04657581 -0.00023914
   0.00127182]
 [ 0.01316391  0.02718937  0.01882807 ... -0.03228774 -0.00484201
  -0.01022142]
 [-0.01892449  0.04762594  0.03378283 ... -0.03411607  0.00606877
   0.00333418]
 [ 0.0475      0.06625011  0.00719741 ... -0.02137133 -0.05288657
  -0.03651163]
 [ 0.00091584  0.07381378 -0.01144866 ... -0.03380296  0.01246412
   0.06100812]]


In [ ]:
print(latent_topics[:2])

[[ 0.00534919 -0.00192709 -0.00189574  0.0025188   0.00031153  0.00232007
   0.00413228  0.00204923 -0.00094213  0.00229298  0.00314967  0.00196387
   0.00207846  0.00046774 -0.00035579 -0.00049944 -0.00123874  0.00647422
   0.00073084  0.00259619  0.00375475  0.00471274  0.00505178  0.00117801
   0.00245866 -0.00077487 -0.00086201  0.000315    0.00044301 -0.00101963
  -0.00325242 -0.00240593  0.00405029 -0.00435497  0.01550045  0.00329793
   0.00612442  0.00151826  0.00766863 -0.00891806  0.0164741   0.00031736
   0.00418337 -0.00375973  0.01195023  0.00025334 -0.00703579 -0.00552716
   0.00325811  0.0039912 ]
 [ 0.01758215 -0.00816714 -0.01597691  0.02994953  0.0085223   0.04704055
   0.00605871  0.00714621 -0.00461238  0.00046148 -0.01363052  0.0203626
   0.02480049  0.0191942  -0.00350927 -0.00210079 -0.01276695  0.02390706
   0.01683949  0.0160779   0.01041365  0.01305636  0.01080212 -0.01564392
   0.01855075  0.02107487 -0.00429233  0.01655651 -0.00122229  0.0156323
  -0.04086212

In [ ]:
news_features.head()

,news_id,subclustered_category,ctr_norm
0,26629,lifestyleroyals,-0.153225
1,45241,weightloss,-0.153225
2,49591,CounterTerrorism_MiddleEastConflict,-0.153225
3,28765,health_general,-0.153225
4,28005,medical,-0.153225


In [ ]:
train_user_features.head()

,impression_id,user_id,time,history,pos_id,neg_ids,top_cat_1,top_cat_2,top_cat_3,recent_fav_cat,history_length,diversity_ratio,concentration_ratio,diversity_cluster,diversity_level,interest_cluster,interest_cluster_name
0,1,1,2019-11-11 09:05:58,"[26659, 7649, 10157, 13853, 11735, 5063, 22437...",17918,"[35306, 35306, 35306, 35306]",tvnews,2020Election_LegislativeReform,MLB_Awards_Rankings,tvnews,9,0.777778,0.222222,0,High Diversity,0,Entertainment & Sports Readers
1,2,2,2019-11-12 18:11:30,"[19437, 27124, 48998, 22350, 37334, 22912, 447...",37331,"[24166, 6548, 31029, 4834]",ViolentCrime,PolicePursuits_DrugCrime,US_ColdCases_PoliceMisconduct,travelnews,82,0.256098,0.329268,1,Medium Diversity,0,Entertainment & Sports Readers
2,4,4,2019-11-11 05:28:05,"[49779, 14969, 23815, 48261, 33954, 15429, 511...",43802,"[35306, 13283, 35306, 13283]",tv-celebrity,markets,basketball_nba,tv-celebrity,10,0.700000,0.200000,1,Medium Diversity,0,Entertainment & Sports Readers
3,5,5,2019-11-12 16:11:21,"[26331, 5983, 16352, 33322]",20105,"[12838, 29009, 13483, 35484]",autos_general,travelnews,ClimateInfrastructure_Pollution,autos_general,4,1.000000,0.250000,0,High Diversity,0,Entertainment & Sports Readers
4,6,6,2019-11-11 18:52:13,"[5143, 38345, 35829, 24011, 7704, 47885, 43992...",5498,"[3319, 1557, 30009, 30009]",US_ColdCases_PoliceMisconduct,CounterTerrorism_MiddleEastConflict,newsgoodnews,newsgoodnews,36,0.500000,0.194444,1,Medium Diversity,0,Entertainment & Sports Readers


In [ ]:
train_context_features.head()

,impression_id,user_id,time,history,pos_id,neg_ids,hour_sin,hour_cos,habit_score,candidate_articles,labels,recency_scores
31543,35263,1,2019-11-09 05:59:43,"[26659, 7649, 10157, 13853, 11735, 5063, 22437...",515,"[29599, 44636, 17569, 2]",0.965926,2.588190e-01,0.000000,"[515, 17569, 2, 44636, 29599]","[1, 0, 0, 0, 0]","[1.0, 1.0, 1.0, 1.0, 1.0]"
0,1,1,2019-11-11 09:05:58,"[26659, 7649, 10157, 13853, 11735, 5063, 22437...",17918,"[35306, 35306, 35306, 35306]",0.707107,-7.071068e-01,0.687563,"[35306, 35306, 35306, 17918, 35306]","[0, 0, 0, 1, 0]","[0.07047630234333706, 0.07047630234333706, 0.0..."
18492,20676,2,2019-11-09 05:28:51,"[19437, 27124, 48998, 22350, 37334, 22912, 447...",44926,"[858, 46065, 33826, 18405]",0.965926,2.588190e-01,0.000000,"[18405, 44926, 46065, 33826, 858]","[0, 1, 0, 0, 0]","[1.0, 1.0, 1.0, 1.0, 1.0]"
6124,6673,2,2019-11-09 18:21:10,"[19437, 27124, 48998, 22350, 37334, 22912, 447...",49940,"[10538, 35352, 10348, 34170]",-1.000000,-1.836970e-16,-0.974058,"[10538, 49940, 10348, 35352, 34170]","[0, 1, 0, 0, 0]","[1.0, 0.085166784953868, 1.0, 1.0, 0.125896135..."
87757,97245,2,2019-11-10 05:43:35,"[19437, 27124, 48998, 22350, 37334, 22912, 447...",4154,"[49295, 39215, 22978, 19724]",0.965926,2.588190e-01,-0.986584,"[49295, 22978, 4154, 19724, 39215]","[0, 0, 1, 0, 0]","[0.04037413363838234, 0.060132291040288624, 1...."


In [ ]:
val_context_features.head()

,impression_id,user_id,time,history,pos_ids,neg_ids,pos_count,neg_count,hour_sin,hour_cos,habit_score,candidate_articles,labels,recency_scores
31184,154837,1,2019-11-13 15:27:40,"[26659, 7649, 10157, 13853, 11735, 5063, 22437...",[40277],"[7216, 37332, 35088, 35647, 26730, 16019, 4605...",1,25,-0.707107,-0.707107,0.000000,"[51214, 32500, 38721, 46883, 136, 45528, 28324...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0.010448077408644623, 0.013180054184667203, 0..."
9116,45065,2,2019-11-13 20:06:29,"[19437, 27124, 48998, 22350, 37334, 22912, 447...",[32500],"[30325, 35647, 46413, 28324, 32031, 37846, 460...",1,10,-0.866025,0.500000,0.000000,"[35647, 32031, 32500, 37846, 46040, 46413, 283...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[1.0, 0.015504877576070804, 0.0124194032476739..."
4725,23421,6,2019-11-13 11:37:01,"[5143, 38345, 35829, 24011, 7704, 47885, 43992...",[6671],"[26734, 30573, 6474, 19179]",1,4,0.258819,-0.965926,0.000000,"[6474, 26734, 30573, 6671, 19179]","[0, 0, 0, 1, 0]","[1.0, 1.0, 1.0, 0.021461786097531896, 0.019066..."
10073,49849,9,2019-11-13 09:44:50,"[21730, 35908, 35939, 2162, 7695, 42513, 21573...",[13656],[35088],1,1,0.707107,-0.707107,0.000000,"[13656, 35088]","[1, 0]","[0.059512001586986704, 0.019186799481956413]"
0,9,9,2019-11-13 10:13:02,"[21730, 35908, 35939, 2162, 7695, 42513, 21573...",[35647],"[41034, 7216, 8261]",1,3,0.500000,-0.866025,0.992439,"[35647, 41034, 7216, 8261]","[1, 0, 0, 0]","[1.0, 0.021084319708567847, 1.0, 0.03130489225..."


In [ ]:
train_interactions.sample(5)

,impression_id,user_id,time,history,pos_id,neg_ids,candidate_news,labels,knowledge_match_score_kg,knowledge_match_score_semantic,knowledge_match_score_combined
58094,64426,1610,2019-11-12 15:03:39,"[33260, 20070, 20070, 19224, 1083, 22350, 1253...",12838,"[6956, 31145, 10439, 37331]","[N12838, N6956, N31145, N10439, N37331]","[1, 0, 0, 0, 0]","[0.5089555382728577, 0.19136014580726624, 0.64...","[0.5806270241737366, 0.5304443836212158, 0.600...","[0.5447912812232971, 0.36090226471424103, 0.62..."
91457,101365,1311,2019-11-11 11:12:28,"[574, 44601, 10187, 1083, 10245, 28726, 21392,...",20609,"[35743, 34585, 43202, 33248]","[N20609, N35743, N34585, N43202, N33248]","[1, 0, 0, 0, 0]","[0.27616605162620544, 0.5211618542671204, 0.68...","[0.5346004366874695, 0.7120398283004761, 0.726...","[0.40538324415683746, 0.6166008412837982, 0.70..."
116304,128901,4952,2019-11-12 11:53:10,"[1504, 47313, 38131, 3934, 5311, 36496, 47149,...",19088,"[28919, 27781, 13118, 35841]","[N19088, N28919, N27781, N13118, N35841]","[1, 0, 0, 0, 0]","[0.5636128187179565, 0.6535053253173828, 0.0, ...","[0.7854968905448914, 0.7064304351806641, 0.0, ...","[0.674554854631424, 0.6799678802490234, 0.0, 0..."
139730,154889,32062,2019-11-12 07:50:00,"[43715, 37947, 25343, 42860, 373, 20925, 11383...",40853,"[50460, 5973, 3397, 35297]","[N40853, N50460, N5973, N3397, N35297]","[1, 0, 0, 0, 0]","[0.0, 0.0, 0.0, 0.6103445291519165, 0.24328885...","[0.0, 0.0, 0.0, 0.480444073677063, 0.0]","[0.0, 0.0, 0.0, 0.5453943014144897, 0.12164442..."
106642,118160,24265,2019-11-11 08:32:58,"[41614, 21140, 35760, 15429, 20387, 32772, 421...",17918,"[27660, 43557, 44164, 6869]","[N17918, N27660, N43557, N44164, N6869]","[1, 0, 0, 0, 0]","[0.24220407009124756, 0.37967318296432495, 0.0...","[0.6521673798561096, 0.763385534286499, 0.0, 0...","[0.4471857249736786, 0.571529358625412, 0.0, 0..."


In [ ]:
val_interactions.sample(5)

,impression_id,user_id,time,history,pos_ids,neg_ids,pos_count,neg_count,candidate_news,labels,knowledge_match_score_kg,knowledge_match_score_semantic,knowledge_match_score_combined
19671,97503,21347,2019-11-13 15:44:03,"[43715, 2307, 29819, 14974, 12764, 34706, 2979...",[340],[42556],1,1,"[N340, N42556]","[1, 0]","[0.37648823857307434, 0.0]","[0.6935644745826721, 0.35319116711616516]","[0.5350263565778732, 0.17659558355808258]"
11706,57702,31724,2019-11-13 14:10:22,"[41276, 110, 30796, 11383, 13888, 22535, 8981,...",[37846],"[15532, 37332, 49296, 29213, 46230, 35647, 251...",1,43,"[N37846, N15532, N37332, N49296, N29213, N4623...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.4728562533855438, 0.0, 0.0, 0.5225834846496...","[0.49233949184417725, 0.0, 0.360076367855072, ...","[0.48259787261486053, 0.0, 0.180038183927536, ..."
21334,105740,42997,2019-11-13 11:09:51,"[27659, 23543, 7501, 44466, 3138, 28522, 8101,...",[29130],"[43320, 12591, 827, 49262, 10975, 29617, 35955...",1,86,"[N29130, N43320, N12591, N827, N49262, N10975,...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.458933562040329, 0.6320841312408447, 0.3678...","[0.763134241104126, 0.6080658435821533, 0.6834...","[0.6110339015722275, 0.620074987411499, 0.5256..."
28488,141268,28153,2019-11-13 11:59:47,"[40684, 20494, 44601, 14229, 8530, 36841, 2191...",[1587],"[48212, 9063, 15532, 23827, 22273, 35088, 1221...",1,93,"[N1587, N48212, N9063, N15532, N23827, N22273,...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.7444594502449036, 0.48755648732185364, 0.0,...","[0.8006076812744141, 0.6081730127334595, 0.0, ...","[0.7725335657596588, 0.5478647500276566, 0.0, ..."
10724,52959,30217,2019-11-13 10:49:43,"[144, 27037, 32091, 16204, 43353, 47900, 29980...",[8261],[13656],1,1,"[N8261, N13656]","[1, 0]","[0.6192194819450378, 0.0]","[0.5674149394035339, 0.0]","[0.5933172106742859, 0.0]"


## 1.1 Model Definition
The model scores one (`user`, `candidate_news`) pair at a time and outputs a single scalar. The listwise ranking later on compares all 5 candidate news articles (1 clicked, 4 non-clicked) within an impression.

In [ ]:
class NewsMLP(nn.Module):
    """
    Scores a single (user, news) candidate pair → scalar relevance score.
    Listwise loss is applied outside the model over all 5 scores per group.
    """
    def __init__(
        self,
        user_numerical_dense_dim,
        user_categorical_vocab_sizes,   # dict {col: vocab_size}
        news_dense_dim,
        news_categorical_vocab_size,
        bert_dim=768,
        topic_dim=50,
        entity_dim=100,
        context_dense_dim=4,
        hidden_dims=[128, 64, 32],
        embed_dim=128,
    ):
        super(NewsMLP, self).__init__()

        # ── User features ────────────────────────────────────────────────────
        self.user_numerical_dense_proj = nn.Linear(user_numerical_dense_dim, embed_dim)

        self.user_categorical_embeddings = nn.ModuleDict({
            col: nn.Embedding(vocab_size + 1, embed_dim)
            for col, vocab_size in user_categorical_vocab_sizes.items()
        })

        # ── News features ─────────────────────────────────────────────────────
        self.news_dense_proj       = nn.Linear(news_dense_dim, embed_dim)
        self.news_category_embedding = nn.Embedding(news_categorical_vocab_size + 1, embed_dim)
        self.bert_proj             = nn.Linear(bert_dim, embed_dim)
        self.topic_proj            = nn.Linear(topic_dim, embed_dim)
        self.entity_proj           = nn.Linear(entity_dim, embed_dim)

        # ── Interaction features ──────────────────────────────────────────────
        self.knowledge_proj = nn.Linear(1, embed_dim)

        # ── Context features ──────────────────────────────────────────────────
        self.context_proj = nn.Linear(context_dense_dim, embed_dim)

        # ── MLP ───────────────────────────────────────────────────────────────
        # Features: user_numerical(1) + user_categorical(N) +
        #           news_numerical(1) + news_category(1) +
        #           bert(1) + topic(1) + entity(1) + context(1) + knowledge(1)
        num_embedded_features = 1 + len(user_categorical_vocab_sizes) + 1 + 1 + 1 + 1 + 1 + 1 + 1
        total_input_dim = embed_dim * num_embedded_features

        self.fc1    = nn.Linear(total_input_dim, hidden_dims[0])
        self.fc2    = nn.Linear(hidden_dims[0],  hidden_dims[1])
        self.fc3    = nn.Linear(hidden_dims[1],  hidden_dims[2])
        # Single scalar output — listwise competition happens at the loss level
        self.output = nn.Linear(hidden_dims[2], 1)

        self.relu = nn.ReLU()

    def forward(
        self,
        user_numerical_dense,    # [B, user_numerical_dense_dim]
        user_categorical_input,  # dict {col: [B]}
        news_numerical_dense,    # [B, news_dense_dim]
        news_category_indices,   # [B]
        bert_embed,              # [B, bert_dim]
        topic_embed,             # [B, topic_dim]
        entity_embed,            # [B, entity_dim]
        knowledge_score,         # [B, 1]
        context_dense,           # [B, 4]
    ):
        # User
        user_numerical_vec   = self.relu(self.user_numerical_dense_proj(user_numerical_dense))
        user_categorical_vecs = [
            self.user_categorical_embeddings[col](indices.long())
            for col, indices in user_categorical_input.items()
        ]
        user_combined_vec = torch.cat([user_numerical_vec] + user_categorical_vecs, dim=1)

        # News
        news_numerical_vec  = self.relu(self.news_dense_proj(news_numerical_dense))
        news_category_vec   = self.relu(self.news_category_embedding(news_category_indices.long()))
        bert_vec            = self.relu(self.bert_proj(bert_embed))
        topic_vec           = self.relu(self.topic_proj(topic_embed))
        entity_vec          = self.relu(self.entity_proj(entity_embed))
        news_combined_vec   = torch.cat(
            [news_numerical_vec, news_category_vec, bert_vec, topic_vec, entity_vec], dim=1
        )

        # Interaction
        knowledge_vec = self.relu(self.knowledge_proj(knowledge_score))

        # Context
        context_vec = self.relu(self.context_proj(context_dense))

        # Concatenate everything
        x = torch.cat([user_combined_vec, news_combined_vec, knowledge_vec, context_vec], dim=1) # Added context_vec

        # MLP
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))

        score = self.output(x)   # [B, 1]
        return score.squeeze(1)  # [B]

#2 Input Preparation

News Features
- category
- ctr

User features
- history length
- recent favourite category
- top category 1
- top category 2
- top category 3
- diversity cluster
- category cluster

Context features
- hour of day
- recency score
- habit score

Interaction features
- knowledge match score

##2.1 User Features

We process the features listed below.

*   Total Historical Clicks
*   Receent Favourite Category
*   Top Category 1
*   Top Category 2
*   Top Category 3
*   Diversity Cluster
*   Category Cluster



In [ ]:
# Standardise column names: train uses 'pos_id', val/test use 'pos_ids'
# Make everything use 'pos_id' (scalar) and 'neg_ids' (list of 4)
if 'pos_ids' in train_user_features.columns:
    train_user_features['pos_id'] = train_user_features['pos_ids'].apply(
        lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x
    )
if 'pos_ids' in val_user_features.columns:
    val_user_features['pos_id'] = val_user_features['pos_ids'].apply(
        lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x
    )
if 'pos_ids' in test_user_features.columns:
    test_user_features['pos_id'] = test_user_features['pos_ids'].apply(
        lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x
    )

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

numerical_user_features = [
    'history_length',
    'diversity_ratio',
    'concentration_ratio'
]

categorical_user_features = [
    'recent_fav_cat',
    'top_cat_1',
    'top_cat_2',
    'top_cat_3',
    'diversity_cluster', # Moved here
    'interest_cluster'   # Moved here
]

# Filter features that actually exist in the DataFrame
selected_numerical_user_features = [f for f in numerical_user_features if f in train_user_features.columns]
selected_categorical_user_features = [f for f in categorical_user_features if f in train_user_features.columns]

print(f"Numerical user features: {selected_numerical_user_features}")
print(f"Categorical user features: {selected_categorical_user_features}")

Numerical user features: ['history_length', 'diversity_ratio', 'concentration_ratio']
Categorical user features: ['recent_fav_cat', 'top_cat_1', 'top_cat_2', 'top_cat_3', 'diversity_cluster', 'interest_cluster']


Below is the preprocessing function for user features.

In [ ]:
def preprocess_user_features(df, numerical_cols, categorical_cols, categorical_mappers=None, scaler=None):

    # 1. Extract numerical features
    user_numerical_data = df[numerical_cols].values.astype(np.float32)

    # 2. Log Transform history_length to fix the heavy-tail skew
    if 'history_length' in numerical_cols:
        hl_idx = numerical_cols.index('history_length')
        user_numerical_data[:, hl_idx] = np.log1p(user_numerical_data[:, hl_idx])

    # 3. Apply Standard Scaler
    if scaler is None:
        scaler = StandardScaler()
        user_numerical_data = scaler.fit_transform(user_numerical_data).astype(np.float32)
    else:
        user_numerical_data = scaler.transform(user_numerical_data).astype(np.float32)

    col = user_numerical_data[:, hl_idx]
    print("Exact Min:", col.min())
    print("Exact Max:", col.max())
    print("Mean:", col.mean())
    print("Std:", col.std())

    # 4. Categorical Encoding
    user_categorical_encoded_df = pd.DataFrame(index=df.index)
    current_categorical_mappers = {}
    vocab_sizes = {}

    if categorical_mappers is None: # Learn mappers from training data
        for col in categorical_cols:
            codes, uniques = pd.factorize(df[col].astype(str))
            user_categorical_encoded_df[col] = codes
            current_categorical_mappers[col] = {category: i for i, category in enumerate(uniques)}
            vocab_sizes[col] = len(uniques)
        final_mappers_to_return = current_categorical_mappers
    else: # Apply existing mappers to validation/test data
        final_mappers_to_return = categorical_mappers
        for col in categorical_cols:
            mapper = categorical_mappers[col]
            mapped_codes = df[col].astype(str).map(mapper).fillna(len(mapper)).astype(int)
            user_categorical_encoded_df[col] = mapped_codes
            vocab_sizes[col] = len(mapper) + 1

    # Return the scaler as the 5th variable
    return user_numerical_data, user_categorical_encoded_df, final_mappers_to_return, vocab_sizes, scaler

We apply the preprocessing function to our **training** data.

In [ ]:
# Process training data and capture the categorical mappers, vocab sizes, AND scaler
train_user_numerical_input, train_user_categorical_input, user_category_mappers, user_vocab_sizes, trained_scaler = preprocess_user_features(
    train_user_features,
    selected_numerical_user_features,
    selected_categorical_user_features
)

user_numerical_dense_dim = train_user_numerical_input.shape[1]

print(f"Shape of train_user_numerical_input: {train_user_numerical_input.shape}")
print(f"Shape of train_user_categorical_input: {train_user_categorical_input.shape}")
print(f"User numerical dense dimension: {user_numerical_dense_dim}")
print(f"Learned categorical mappers (first 5 for each category):")
for col, mapper in user_category_mappers.items():
    print(f"  {col}: {list(mapper.items())[:5]} ... (total {len(mapper)})")
print(f"Learned vocabulary sizes: {user_vocab_sizes}")

Exact Min: -2.7185128
Exact Max: 2.848689
Mean: 3.8266084e-09
Std: 1.0
Shape of train_user_numerical_input: (141558, 3)
Shape of train_user_categorical_input: (141558, 6)
User numerical dense dimension: 3
Learned categorical mappers (first 5 for each category):
  recent_fav_cat: [('tvnews', 0), ('travelnews', 1), ('tv-celebrity', 2), ('autos_general', 3), ('newsgoodnews', 4)] ... (total 96)
  top_cat_1: [('tvnews', 0), ('ViolentCrime', 1), ('tv-celebrity', 2), ('autos_general', 3), ('US_ColdCases_PoliceMisconduct', 4)] ... (total 96)
  top_cat_2: [('2020Election_LegislativeReform', 0), ('PolicePursuits_DrugCrime', 1), ('markets', 2), ('travelnews', 3), ('CounterTerrorism_MiddleEastConflict', 4)] ... (total 96)
  top_cat_3: [('MLB_Awards_Rankings', 0), ('US_ColdCases_PoliceMisconduct', 1), ('basketball_nba', 2), ('ClimateInfrastructure_Pollution', 3), ('newsgoodnews', 4)] ... (total 96)
  diversity_cluster: [('0', 0), ('1', 1), ('2', 2), ('-1', 3)] ... (total 4)
  interest_cluster: [('0

We apply the preprocessing function to our **validation** and **testing**  data.


In [ ]:
# Pass the trained_scaler in to properly normalize validation and test data
val_user_numerical_input, val_user_categorical_input, _, _, _ = preprocess_user_features(
    val_user_features,
    selected_numerical_user_features,
    selected_categorical_user_features,
    categorical_mappers=user_category_mappers,
    scaler=trained_scaler # Added scaler
)

test_user_numerical_input, test_user_categorical_input, _, _, _ = preprocess_user_features(
    test_user_features,
    selected_numerical_user_features,
    selected_categorical_user_features,
    categorical_mappers=user_category_mappers,
    scaler=trained_scaler # Added scaler
)

print(f"Shape of val_user_numerical_input: {val_user_numerical_input.shape}")
print(f"Shape of val_user_categorical_input: {val_user_categorical_input.shape}")
print(f"Shape of test_user_numerical_input: {test_user_numerical_input.shape}")
print(f"Shape of test_user_categorical_input: {test_user_categorical_input.shape}")

Exact Min: -2.7185128
Exact Max: 2.848689
Mean: -0.10478702
Std: 0.989327
Exact Min: -2.7185128
Exact Max: 2.848689
Mean: -0.11257638
Std: 0.99369144
Shape of val_user_numerical_input: (31624, 3)
Shape of val_user_categorical_input: (31624, 6)
Shape of test_user_numerical_input: (30270, 3)
Shape of test_user_categorical_input: (30270, 6)


We define the function below which generates the weight (`ctr_weight`) to be applied for the news feature `ctr_norm` (article global popularity). This is to allow model to rely more heavily on `ctr_norm` feature for ranking articles if the user is a cold-start user or has very few articles in their user history.

* This function will **not** be applied for the baseline model generated from Steps 1 to 6. It is an **additional feature** that will be incorporated into the **final model** in *Section 10*.

Note: See *Section 7-8* for more details.

In [ ]:
# ── Cold-start CTR weights ────────────────────────────────────────────────────
# At history_length=0  → weight = 1.0 (full CTR reliance)
# At history_length=11 → weight = 0.05 (5% CTR influence)
# At history_length=20 → weight ≈ 0.002 (negligible)

LAMBDA = 0.4

def compute_ctr_weights(user_features_df, lam=LAMBDA):
    history = user_features_df['history_length'].values.astype(np.float32)
    weights = np.exp(-lam * history).reshape(-1, 1).astype(np.float32)
    return torch.from_numpy(weights)

train_ctr_weights = compute_ctr_weights(train_user_features)
val_ctr_weights   = compute_ctr_weights(val_user_features)
test_ctr_weights  = compute_ctr_weights(test_user_features)

print(f"train_ctr_weights shape : {train_ctr_weights.shape}")
print(f"val_ctr_weights shape   : {val_ctr_weights.shape}")
print(f"Sample weights (first 5): {train_ctr_weights[:5].squeeze().tolist()}")

# Sanity check — spot check a cold-start user (history_length=0) and active user
cold_start_mask = train_user_features['history_length'] == 0
active_mask     = train_user_features['history_length'] >= 20
print(f"\nCold-start users  → avg CTR weight: {train_ctr_weights[cold_start_mask].mean():.4f}")
print(f"Active users (20+) → avg CTR weight: {train_ctr_weights[active_mask].mean():.4f}")

train_ctr_weights shape : torch.Size([141558, 1])
val_ctr_weights shape   : torch.Size([31624, 1])
Sample weights (first 5): [0.02732371911406517, 5.690380468665072e-15, 0.018315639346837997, 0.2018965184688568, 5.573900807576138e-07]

Cold-start users  → avg CTR weight: 1.0000
Active users (20+) → avg CTR weight: 0.0000


In [ ]:
# Convert user numerical features to tensors
train_user_numerical_tensor = torch.from_numpy(train_user_numerical_input)
val_user_numerical_tensor = torch.from_numpy(val_user_numerical_input)
test_user_numerical_tensor = torch.from_numpy(test_user_numerical_input)

# Convert user categorical features (DataFrame of integer codes) to tensors
train_user_categorical_tensors = {col: torch.from_numpy(train_user_categorical_input[col].values) for col in selected_categorical_user_features}
val_user_categorical_tensors = {col: torch.from_numpy(val_user_categorical_input[col].values) for col in selected_categorical_user_features}
test_user_categorical_tensors = {col: torch.from_numpy(test_user_categorical_input[col].values) for col in selected_categorical_user_features}

print(f"Train user numerical tensor shape: {train_user_numerical_tensor.shape}")
print(f"Train user categorical tensors (example): {train_user_categorical_tensors[selected_categorical_user_features[0]].shape}")

Train user numerical tensor shape: torch.Size([141558, 3])
Train user categorical tensors (example): torch.Size([141558])


##2.2 News Features

*   Category
*   Popularity `ctr_norm`
*   Title and Abstract BERT Embedding
*   Latent Topics Embedding
*   Entity Embedding



In [ ]:
# Extract numerical news features (ctr_norm)
news_numerical_input = news_features["ctr_norm"].values.reshape(-1, 1).astype(np.float32)

# Integer encode 'subclustered_category' for news features
news_category_codes, news_category_uniques = pd.factorize(news_features['subclustered_category'].astype(str))
news_categorical_input = news_category_codes
news_categorical_vocab_size = len(news_category_uniques)

print(f"Shape of news_numerical_input: {news_numerical_input.shape}")
print(f"Shape of news_categorical_input: {news_categorical_input.shape}")
print(f"News categorical vocabulary size: {news_categorical_vocab_size}")

# news_dense_dim is 1 for ctr_norm
news_dense_dim = news_numerical_input.shape[1]

Shape of news_numerical_input: (51282, 1)
Shape of news_categorical_input: (51282,)
News categorical vocabulary size: 95


In [ ]:
# Convert news numerical (ctr_norm) and categorical (subclustered_category) to tensors
news_numerical_tensor = torch.from_numpy(news_numerical_input)
news_categorical_tensor = torch.from_numpy(news_categorical_input)

print(f"News numerical tensor shape: {news_numerical_tensor.shape}")
print(f"News categorical tensor shape: {news_categorical_tensor.shape}")

News numerical tensor shape: torch.Size([51282, 1])
News categorical tensor shape: torch.Size([51282])


##2.3 Interaction Features

*  Knowledge Match Score


In [ ]:
# knowledge_match_score_combined is a list of 5 scores per impression
# Shape will be [N, 5, 1]
train_knowledge_score = torch.tensor(
    np.stack(train_interactions['knowledge_match_score_combined'].values),
    dtype=torch.float32
).unsqueeze(2)  # [N, 5, 1]

# val and test have variable-length candidate list per impression
def build_knowledge_lookup(interactions_df):
    """Returns {(impression_id, news_id): score} for variable-length groups."""
    lookup = {}
    for _, row in interactions_df.iterrows():
        imp_id      = row['impression_id']
        candidates  = row['candidate_news']   # e.g. ['N13656', 'N25555', ...]
        scores      = row['knowledge_match_score_combined']
        for news_id, score in zip(candidates, scores):
            # Strip the 'N' prefix to match your news_id format if needed
            lookup[(imp_id, news_id)] = score
    return lookup

val_knowledge_lookup  = build_knowledge_lookup(val_interactions)
test_knowledge_lookup = build_knowledge_lookup(test_interactions)

##2.4 Context Features
*  Hour of the Day
*  Recency of News Article
*  Habit Score `habit_score`



In [ ]:
train_context_scalar = torch.tensor(
    train_context_features[['hour_sin', 'hour_cos', 'habit_score']].values.astype(np.float32)
)  # [N_train, 3]

val_context_scalar = torch.tensor(
    val_context_features[['hour_sin', 'hour_cos', 'habit_score']].values.astype(np.float32)
)  # [N_val, 3]

test_context_scalar = torch.tensor(
    test_context_features[['hour_sin', 'hour_cos', 'habit_score']].values.astype(np.float32)
)  # [N_test, 3]

# ── Recency score: train has fixed 5 candidates, stack into tensor ────────────
train_recency_score = torch.tensor(
    np.stack(train_context_features['recency_scores'].values),
    dtype=torch.float32
)  # [N_train, 5]

# ── Recency score: val/test have variable candidates, use lookup ──────────────
def build_recency_lookup(context_df):
    """Returns {(impression_id, news_id): recency_scores}"""
    lookup = {}
    for _, row in context_df.iterrows():
        imp_id     = row['impression_id']
        candidates = row['candidate_articles'] # Corrected column name
        scores     = row['recency_scores']
        for news_id, score in zip(candidates, scores):
            lookup[(imp_id, news_id)] = score
    return lookup

val_recency_lookup  = build_recency_lookup(val_context_features)
test_recency_lookup = build_recency_lookup(test_context_features)

print(f"train_context_scalar shape:  {train_context_scalar.shape}")
print(f"train_recency_score shape:   {train_recency_score.shape}")
print(f"val recency lookup size:     {len(val_recency_lookup)}")

train_context_scalar shape:  torch.Size([141558, 3])
train_recency_score shape:   torch.Size([141558, 5])
val recency lookup size:     1215336


##2.5 Embeddings

In [ ]:
from sklearn.decomposition import PCA

entity_embed_dim = 100

kg_title_matrix      = np.stack(news_entities["kg_title_emb"].values)
kg_abstract_matrix   = np.stack(news_entities["kg_abstract_emb"].values)
ner_title_matrix     = np.stack(news_entities["ner_title_emb"].values)
ner_abstract_matrix  = np.stack(news_entities["ner_abstract_emb"].values)
flags_matrix         = news_entities[
    ["kg_title_flag", "ner_title_flag", "kg_abstract_flag", "ner_abstract_flag"]
].values.astype(np.float32)

pca_title    = PCA(n_components=entity_embed_dim)
pca_abstract = PCA(n_components=entity_embed_dim)
projected_ner_title_matrix    = pca_title.fit_transform(ner_title_matrix).astype(np.float32)
projected_ner_abstract_matrix = pca_abstract.fit_transform(ner_abstract_matrix).astype(np.float32)

title_entity_embed   = np.where(flags_matrix[:, 0].reshape(-1, 1) == 1, kg_title_matrix,   projected_ner_title_matrix)
abstract_entity_embed = np.where(flags_matrix[:, 2].reshape(-1, 1) == 1, kg_abstract_matrix, projected_ner_abstract_matrix)
entity_embed_np      = (title_entity_embed + abstract_entity_embed).astype(np.float32)

entity_embed_tensor     = torch.from_numpy(entity_embed_np)
bert_embeddings_tensor  = torch.from_numpy(bert_embeddings)
topics_tensor           = torch.from_numpy(latent_topics)

# Global lookup: news_id → integer index into the news feature tensors
news_id_to_idx_map = {nid: idx for idx, nid in enumerate(news_features['news_id'])}

print(f"entity_embed_tensor shape: {entity_embed_tensor.shape}")
print(f"bert_embeddings_tensor shape: {bert_embeddings_tensor.shape}")
print(f"topics_tensor shape: {topics_tensor.shape}")

entity_embed_tensor shape: torch.Size([51282, 100])
bert_embeddings_tensor shape: torch.Size([51282, 768])
topics_tensor shape: torch.Size([51282, 50])


In [ ]:
# ── Align all impression-level data to train_interactions ─────────────────────
# train_interactions is the authoritative impression-level table.
# Merge user features into it so row counts match knowledge_score exactly.

"""
train_user_features = train_user_features.reset_index(drop=True)
train_interactions  = train_interactions.reset_index(drop=True)

# Verify they are the same length before proceeding
assert len(train_user_features) == len(train_interactions), \
    f"Row mismatch: train_user_features={len(train_user_features)}, train_interactions={len(train_interactions)}"

# Do the same for val and test
assert len(val_user_features) == len(val_interactions), \
    f"Row mismatch: val_user_features={len(val_user_features)}, val_interactions={len(val_interactions)}"
assert len(test_user_features) == len(test_interactions), \
    f"Row mismatch: test_user_features={len(test_user_features)}, test_interactions={len(test_interactions)}"
"""

'\ntrain_user_features = train_user_features.reset_index(drop=True)\ntrain_interactions  = train_interactions.reset_index(drop=True)\n\n# Verify they are the same length before proceeding\nassert len(train_user_features) == len(train_interactions),     f"Row mismatch: train_user_features={len(train_user_features)}, train_interactions={len(train_interactions)}"\n\n# Do the same for val and test\nassert len(val_user_features) == len(val_interactions),     f"Row mismatch: val_user_features={len(val_user_features)}, val_interactions={len(val_interactions)}"\nassert len(test_user_features) == len(test_interactions),     f"Row mismatch: test_user_features={len(test_user_features)}, test_interactions={len(test_interactions)}"\n'

#3 Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class NewsListwiseDataset(Dataset):
    """
    Each __getitem__ returns features for ONE impression group:
        - user features (shared across all 5 candidates)
        - news features for the positive candidate  (index 0)
        - news features for each of the 4 negatives (indices 1-4)
        - knowledge score for the impression
        - impression_id
    The positive is always placed at position 0 in the candidate list,
    so the target index for cross-entropy is always 0.
    """
    def __init__(
        self,
        user_numerical_features,       # np/tensor [N, user_num_dim]
        user_categorical_features_dict,# dict {col: tensor [N]}
        pos_news_ids,                  # pd.Series [N]  — one pos per impression
        neg_news_ids,                  # pd.Series [N]  — each cell is a list of 4 neg ids
        news_id_to_idx_map,
        all_news_numerical,            # tensor [M, news_num_dim]
        all_news_categorical,          # tensor [M]
        all_bert_embeddings,           # tensor [M, bert_dim]
        all_topic_embeddings,          # tensor [M, topic_dim]
        all_entity_embeddings,         # tensor [M, entity_dim]
        knowledge_scores,              # tensor [N, 1]
        context_scalar,                # tensor [N, 3]  (hour_sin, hour_cos, habit_score)
        recency_scores,                # tensor [N, 5]
        impression_ids,                # pd.Series [N]
    ):
        self.user_numerical   = user_numerical_features
        self.user_categorical = user_categorical_features_dict
        self.pos_news_ids     = pos_news_ids.reset_index(drop=True)
        self.neg_news_ids     = neg_news_ids.reset_index(drop=True)
        self.news_id_to_idx   = news_id_to_idx_map
        self.all_news_num     = all_news_numerical
        self.all_news_cat     = all_news_categorical
        self.all_bert         = all_bert_embeddings
        self.all_topic        = all_topic_embeddings
        self.all_entity       = all_entity_embeddings
        self.knowledge_score  = knowledge_scores
        self.context_scalar   = context_scalar
        self.recency_scores   = recency_scores
        self.impression_ids   = impression_ids.reset_index(drop=True)

    def __len__(self):
        return len(self.pos_news_ids)

    def _get_news_features(self, news_id):
        idx = self.news_id_to_idx.get(news_id, -1)
        if idx == -1:
            raise ValueError(f"News ID {news_id} not found in news_id_to_idx_map.")
        return (
            self.all_news_num[idx],
            self.all_news_cat[idx],
            self.all_bert[idx],
            self.all_topic[idx],
            self.all_entity[idx],
        )

    def __getitem__(self, idx):
        # ── User features (same for all candidates in this impression) ────────
        user_num  = self.user_numerical[idx]                              # [user_num_dim]
        user_cat  = {col: self.user_categorical[col][idx]
                     for col in self.user_categorical}                    # {col: scalar tensor}

        # ── Positive candidate features ───────────────────────────────────────
        pos_feats = self._get_news_features(self.pos_news_ids[idx])      # tuple of 5 tensors

        # ── Negative candidate features ───────────────────────────────────────
        neg_ids   = self.neg_news_ids[idx]                               # list of 4 ids
        neg_feats = [self._get_news_features(nid) for nid in neg_ids]   # list of 4 × (5 tensors)

        # ── Stack candidates: pos first (index 0), then negs ─────────────────
        # Each of the 5 feature types becomes a tensor of shape [num_candidates, feat_dim]
        all_candidates = [pos_feats] + neg_feats                         # 5 tuples
        cand_news_num  = torch.stack([f[0] for f in all_candidates])    # [5, news_num_dim]
        cand_news_cat  = torch.stack([f[1] for f in all_candidates])    # [5]
        cand_bert      = torch.stack([f[2] for f in all_candidates])    # [5, bert_dim]
        cand_topic     = torch.stack([f[3] for f in all_candidates])    # [5, topic_dim]
        cand_entity    = torch.stack([f[4] for f in all_candidates])    # [5, entity_dim]
        knowledge      = self.knowledge_score[idx]                      # [1]
        recency        = self.recency_scores[idx]                       # [5]  — one per candidate
        ctx_scalar     = self.context_scalar[idx]                       # [3]  — shared across candidates
        impression_id  = self.impression_ids[idx]
        # Combine into [5, 4]: repeat scalar context for each candidate, concat with recency
        ctx_scalar_expanded = ctx_scalar.unsqueeze(0).expand(5, -1)     # [5, 3]
        context = torch.cat([ctx_scalar_expanded, recency.unsqueeze(1)], dim=1)  # [5, 4]

        return (
            user_num,          # [user_num_dim]
            user_cat,          # dict {col: scalar tensor}
            cand_news_num,     # [5, news_num_dim]
            cand_news_cat,     # [5]
            cand_bert,         # [5, bert_dim]
            cand_topic,        # [5, topic_dim]
            cand_entity,       # [5, entity_dim]
            knowledge,         # [1]
            context,           # [5.4]
            impression_id,
        )

class NewsListwiseDatasetVariable(Dataset):
    """
    For val/test where each impression can have varying numbers of
    positives and negatives. Returns all candidates per impression.
    """
    def __init__(
        self,
        user_numerical_features,
        user_categorical_features_dict,
        interactions_df,            # val_interactions or test_interactions
        news_id_to_idx_map,
        all_news_numerical,
        all_news_categorical,
        all_bert_embeddings,
        all_topic_embeddings,
        all_entity_embeddings,
        knowledge_lookup,           # dict {(impression_id, news_id): score}
        recency_lookup,
        context_scalar_tensor      # Renamed this parameter as it holds a tensor, not a lookup dict
    ):
        self.user_numerical   = user_numerical_features
        self.user_categorical = user_categorical_features_dict
        self.interactions     = interactions_df.reset_index(drop=True)
        self.news_id_to_idx   = news_id_to_idx_map
        self.all_news_num     = all_news_numerical
        self.all_news_cat     = all_news_categorical
        self.all_bert         = all_bert_embeddings
        self.all_topic        = all_topic_embeddings
        self.all_entity       = all_entity_embeddings
        self.knowledge_lookup = knowledge_lookup
        self.recency_lookup   = recency_lookup
        self.context_scalar_tensor = context_scalar_tensor # Store the [N,3] tensor here

    def __len__(self):
        return len(self.interactions)

    def _get_news_features(self, news_id):
        # news_id may come as 'N13656' or integer — normalise as needed
        if isinstance(news_id, str) and news_id.startswith('N'):
            news_id = int(news_id[1:]) # Strip 'N' and convert to int
        idx = self.news_id_to_idx.get(news_id, -1)
        if idx == -1:
            raise ValueError(f"News ID {news_id} not found.")
        return (
            self.all_news_num[idx],
            self.all_news_cat[idx],
            self.all_bert[idx],
            self.all_topic[idx],
            self.all_entity[idx],
        )

    def __getitem__(self, idx):
        row           = self.interactions.iloc[idx]
        impression_id = row['impression_id']
        candidates    = row['candidate_news']   # full list, pos + neg
        labels        = row['labels']           # e.g. [1, 1, 0, 0, 0, ...]

        user_num = self.user_numerical[idx]
        user_cat = {col: self.user_categorical[col][idx] for col in self.user_categorical}

        # Get features and knowledge score for every candidate
        all_feats = [self._get_news_features(nid) for nid in candidates]
        cand_news_num = torch.stack([f[0] for f in all_feats])   # [C, news_num_dim]
        cand_news_cat = torch.stack([f[1] for f in all_feats])   # [C]
        cand_bert     = torch.stack([f[2] for f in all_feats])   # [C, bert_dim]
        cand_topic    = torch.stack([f[3] for f in all_feats])   # [C, topic_dim]
        cand_entity   = torch.stack([f[4] for f in all_feats])   # [C, entity_dim]

        # Get the scalar context features for this impression
        ctx_scalar = self.context_scalar_tensor[idx]                          # [3]
        ctx_scalar_expanded = ctx_scalar.unsqueeze(0).expand(len(candidates), -1)  # [C, 3]

        # Get the recency score for each candidate using the lookup
        recency = torch.tensor(
            [self.recency_lookup.get((impression_id, nid), 0.0) for nid in candidates],
            dtype=torch.float32
        ).unsqueeze(1)                                                  # [C, 1]

        # Combine scalar context features and recency into the final context tensor for the model
        context_features_combined = torch.cat([ctx_scalar_expanded, recency], dim=1) # [C, 4]

        knowledge = torch.tensor(
            [self.knowledge_lookup.get((impression_id, nid), 0.0) for nid in candidates],
            dtype=torch.float32
        ).unsqueeze(1)   # [C, 1]

        labels_tensor = torch.tensor(labels, dtype=torch.float32)  # [C]

        return (
            user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            context_features_combined, # Return the combined context features
            knowledge,
            labels_tensor,
            impression_id,
        )


def listwise_collate_fn(batch):
    """
    Custom collate so the user_categorical dict of tensors is handled correctly.
    Resulting batch tensors have a leading batch dimension B (number of impressions).
    News tensors have shape [B, 5, feat_dim].
    """
    (user_nums, user_cats, cand_news_nums, cand_news_cats,
     cand_berts, cand_topics, cand_entities,
     contexts, knowledge_scores, impression_ids) = zip(*batch)

    user_num_batch       = torch.stack(user_nums)                   # [B, user_num_dim]
    user_cat_batch       = {
        col: torch.stack([uc[col] for uc in user_cats])
        for col in user_cats[0]
    }                                                                # {col: [B]}
    cand_news_num_batch  = torch.stack(cand_news_nums)              # [B, 5, news_num_dim]
    cand_news_cat_batch  = torch.stack(cand_news_cats)              # [B, 5]
    cand_bert_batch      = torch.stack(cand_berts)                  # [B, 5, bert_dim]
    cand_topic_batch     = torch.stack(cand_topics)                 # [B, 5, topic_dim]
    cand_entity_batch    = torch.stack(cand_entities)               # [B, 5, entity_dim]
    knowledge_batch      = torch.stack(knowledge_scores)            # [B, 1]
    context_batch        = torch.stack(contexts)                    # [B, 5, 4]
    impression_ids_batch = torch.tensor(impression_ids)             # [B]

    return (
        user_num_batch, user_cat_batch,
        cand_news_num_batch, cand_news_cat_batch,
        cand_bert_batch, cand_topic_batch, cand_entity_batch,
        context_batch, knowledge_batch, impression_ids_batch, # Returning context_batch
    )

def variable_collate_fn(batch):
    # batch size is always 1 for variable-length groups
    (user_num, user_cat, cand_news_num, cand_news_cat,
     cand_bert, cand_topic, cand_entity,
     context, knowledge, labels, impression_id) = batch[0]

    return (
        user_num.unsqueeze(0),                                    # [1, user_num_dim]
        {col: user_cat[col].unsqueeze(0) for col in user_cat},   # {col: [1]}
        cand_news_num.unsqueeze(0),                               # [1, C, news_num_dim]
        cand_news_cat.unsqueeze(0),                               # [1, C]
        cand_bert.unsqueeze(0),                                   # [1, C, bert_dim]
        cand_topic.unsqueeze(0),                                  # [1, C, topic_dim]
        cand_entity.unsqueeze(0),                                 # [1, C, entity_dim]
        context.unsqueeze(0),                                     # [1, C, 4]
        knowledge.unsqueeze(0),                                   # [1, C, 1]
        labels.unsqueeze(0),                                      # [1, C]
        torch.tensor([impression_id]),                            # [1]
    )

In [ ]:
# Build datasets
# NOTE: train_interactions must have 'pos_id' (scalar) and 'neg_ids' (list of 4)
#       at the impression level, matching the row order of train_user_features.

train_dataset = NewsListwiseDataset(
    user_numerical_features        = train_user_numerical_tensor,
    user_categorical_features_dict = train_user_categorical_tensors,
    pos_news_ids                   = train_user_features['pos_id'],
    neg_news_ids                   = train_user_features['neg_ids'],
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor,
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_scores               = train_knowledge_score,
    context_scalar                 = train_context_scalar,
    recency_scores                 = train_recency_score,
    impression_ids                 = train_user_features['impression_id'],
)

val_dataset = NewsListwiseDatasetVariable(
    user_numerical_features        = val_user_numerical_tensor,
    user_categorical_features_dict = val_user_categorical_tensors,
    interactions_df                = val_interactions, # Use the full interactions df
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor,
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_lookup               = val_knowledge_lookup, # Use the lookup dict
    recency_lookup                 = val_recency_lookup,
    context_scalar_tensor          = val_context_scalar, # Pass the tensor here
)

test_dataset = NewsListwiseDatasetVariable(
    user_numerical_features        = test_user_numerical_tensor,
    user_categorical_features_dict = test_user_categorical_tensors,
    interactions_df                = test_interactions, # Use the full interactions df
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor,
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_lookup               = test_knowledge_lookup, # Use the lookup dict
    recency_lookup                 = test_recency_lookup,
    context_scalar_tensor          = test_context_scalar, # Pass the tensor here
)

print(f"Train impressions: {len(train_dataset)}")
print(f"Val   impressions: {len(val_dataset)}")
print(f"Test  impressions: {len(test_dataset)}")

Train impressions: 141558
Val   impressions: 31624
Test  impressions: 30270


#4 Initialise Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
model = NewsMLP(
    user_numerical_dense_dim    = user_numerical_dense_dim,
    user_categorical_vocab_sizes= user_vocab_sizes,
    news_dense_dim              = news_dense_dim,
    news_categorical_vocab_size = news_categorical_vocab_size,
    bert_dim                    = bert_embeddings_tensor.shape[1],
    topic_dim                   = topics_tensor.shape[1],
    entity_dim                  = entity_embed_tensor.shape[1],
    context_dense_dim           = 4,
    hidden_dims                 = [128, 64, 32],
    embed_dim                   = 128,
).to(device)

print(model)

NewsMLP(
  (user_numerical_dense_proj): Linear(in_features=3, out_features=128, bias=True)
  (user_categorical_embeddings): ModuleDict(
    (recent_fav_cat): Embedding(97, 128)
    (top_cat_1): Embedding(97, 128)
    (top_cat_2): Embedding(97, 128)
    (top_cat_3): Embedding(97, 128)
    (diversity_cluster): Embedding(5, 128)
    (interest_cluster): Embedding(5, 128)
  )
  (news_dense_proj): Linear(in_features=1, out_features=128, bias=True)
  (news_category_embedding): Embedding(96, 128)
  (bert_proj): Linear(in_features=768, out_features=128, bias=True)
  (topic_proj): Linear(in_features=50, out_features=128, bias=True)
  (entity_proj): Linear(in_features=100, out_features=128, bias=True)
  (knowledge_proj): Linear(in_features=1, out_features=128, bias=True)
  (context_proj): Linear(in_features=4, out_features=128, bias=True)
  (fc1): Linear(in_features=1792, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_

#5 Training

##5.1  Hyperparameters

In [ ]:
# Hyperparameters

learning_rate = 1e-3
batch_size    = 32
num_epochs    = 5

##5.2 DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          collate_fn=listwise_collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False,
                          collate_fn=variable_collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False,
                          collate_fn=variable_collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Train batches: 4424
Val   batches: 31624


Helper Function: Score all candidates within an impression

In [ ]:
def score_candidates(model, user_num, user_cat, cand_news_num, cand_news_cat,
                     cand_bert, cand_topic, cand_entity, context, knowledge, device):
    """
    Revised score_candidates that handles Captum's internal batching.
    """
    # 1. Get dynamic shapes
    # Captum might pass B=50 (steps) for a single impression with C candidates
    B = user_num.shape[0]
    num_cands = cand_news_num.shape[1]

    # 2. Expand User Numerical: [B, dim] -> [B, C, dim] -> [B*C, dim]
    user_num_expanded = user_num.unsqueeze(1).expand(-1, num_cands, -1).reshape(B * num_cands, -1)

    # 3. Expand User Categorical: [B] -> [B, C] -> [B*C]
    user_cat_expanded = {
        col: v.unsqueeze(1).expand(-1, num_cands).flatten()
        for col, v in user_cat.items()
    }

    # 4. Flatten News Features: [B, C, dim] -> [B*C, dim]
    news_num_flattened = cand_news_num.reshape(B * num_cands, -1)
    news_cat_flattened = cand_news_cat.flatten()
    bert_flattened     = cand_bert.reshape(B * num_cands, -1)
    topic_flattened    = cand_topic.reshape(B * num_cands, -1)
    entity_flattened   = cand_entity.reshape(B * num_cands, -1)

    # 5. Flatten Context/Knowledge: [B, C, dim] -> [B*C, dim]
    context_flattened   = context.reshape(B * num_cands, -1)
    knowledge_flattened = knowledge.reshape(B * num_cands, -1)

    # 6. Pass to Model
    # model(user_num, user_cat, news_num, news_cat, bert, topic, entity, knowledge, context)
    flat_scores = model(
        user_num_expanded,
        user_cat_expanded,
        news_num_flattened,
        news_cat_flattened,
        bert_flattened,
        topic_flattened,
        entity_flattened,
        knowledge_flattened,
        context_flattened
    )

    # 7. Reshape back to [B, num_cands]
    return flat_scores.reshape(B, num_cands)

In [ ]:
learning_rate = 1e-3
batch_size    = 32

model = NewsMLP(
    user_numerical_dense_dim     = user_numerical_dense_dim,
    user_categorical_vocab_sizes = user_vocab_sizes,
    news_dense_dim               = news_dense_dim,
    news_categorical_vocab_size  = news_categorical_vocab_size,
    bert_dim                     = bert_embeddings_tensor.shape[1],
    topic_dim                    = topics_tensor.shape[1],
    entity_dim                   = entity_embed_tensor.shape[1],
    context_dense_dim            = 4,
    hidden_dims                  = [128, 64, 32],
    embed_dim                    = 64,
).to(device)

# Rebuild train_loader with tuned batch_size
train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True, collate_fn=listwise_collate_fn)

print(f"Tuned  lr         = {learning_rate}")
print(f"Tuned  batch_size = {batch_size}")
print(f"Tuned  embed_dim  = {64}")

Tuned  lr         = 0.001
Tuned  batch_size = 32
Tuned  embed_dim  = 64


##5.3 Ranking Metrics

* AUC: Captures how well the model can distinguish between a clicked and non-clicked article
* **NDCG@5**:
* **NDCG@10**:
* **MAP**:
* **MRR**:

In [ ]:
def dcg_at_k(r, k):
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.

def ndcg_at_k(r, k):
    dcg_max = dcg_at_k(sorted(r, reverse=True), k)
    if not dcg_max:
        return 0.
    return dcg_at_k(r, k) / dcg_max

def mrr_at_k(y_true, y_score, k=None):
    predictions = sorted(zip(y_score, y_true), key=lambda x: x[0], reverse=True)
    for i, (score, label) in enumerate(predictions):
        if label == 1:
            if k is None or i < k:
                return 1.0 / (i + 1)
            return 0.0
    return 0.0

def average_precision_at_k(y_true, y_score, k=None):
    predictions = sorted(zip(y_score, y_true), key=lambda x: x[0], reverse=True)
    if k is not None:
        predictions = predictions[:k]
    num_relevant, sum_prec = 0, 0.0
    for i, (score, label) in enumerate(predictions):
        if label == 1:
            num_relevant += 1
            sum_prec     += num_relevant / (i + 1)
    return sum_prec / num_relevant if num_relevant else 0.0

def calculate_ranking_metrics(impression_ids, all_labels, all_scores, k_ndcg_5=5, k_ndcg_10=10):
    """
    impression_ids : list of impression IDs (one per impression)
    all_labels     : list of label lists  [[1,0,0,0,0], ...]
    all_scores     : list of score lists  [[s0,s1,s2,s3,s4], ...]
    """
    mrr_scores, map_scores, ndcg_5_scores, ndcg_10_scores = [], [], [], []

    for labels, scores in zip(all_labels, all_scores):
        y_true   = np.array(labels)
        y_score  = np.array(scores)

        if np.sum(y_true) == 0:
            continue

        mrr_scores.append(mrr_at_k(y_true, y_score))
        map_scores.append(average_precision_at_k(y_true, y_score))

        ranked     = sorted(zip(y_score, y_true), key=lambda x: x[0], reverse=True)
        rel_at_rank = [item[1] for item in ranked]
        ndcg_5_scores.append(ndcg_at_k(rel_at_rank, k_ndcg_5))
        ndcg_10_scores.append(ndcg_at_k(rel_at_rank, k_ndcg_10))

    return (
        np.mean(mrr_scores)    if mrr_scores    else 0.,
        np.mean(map_scores)    if map_scores    else 0.,
        np.mean(ndcg_5_scores) if ndcg_5_scores else 0.,
        np.mean(ndcg_10_scores)if ndcg_10_scores else 0.,
    )

## 5.4 Fine-tuning Hyperparameters

In [ ]:
!pip install optuna -q
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # keep output clean

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.4 MB/s eta 0:00:00


In [ ]:
def objective(trial):
    lr        = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    embed_dim = trial.suggest_categorical("embed_dim", [64, 128, 256])
    bs        = trial.suggest_categorical("batch_size", [32, 64, 128])

    # Temporary loader with trial batch size
    trial_loader = DataLoader(train_dataset, batch_size=bs,
                              shuffle=True, collate_fn=listwise_collate_fn)

    # Fresh model with trial embed_dim
    trial_model = NewsMLP(
        user_numerical_dense_dim     = user_numerical_dense_dim,
        user_categorical_vocab_sizes = user_vocab_sizes,
        news_dense_dim               = news_dense_dim,
        news_categorical_vocab_size  = news_categorical_vocab_size,
        bert_dim                     = bert_embeddings_tensor.shape[1],
        topic_dim                    = topics_tensor.shape[1],
        entity_dim                   = entity_embed_tensor.shape[1],
        context_dense_dim            = 4,
        hidden_dims                  = [128, 64, 32],  # fixed
        embed_dim                    = embed_dim,
    ).to(device)

    opt = torch.optim.Adam(trial_model.parameters(), lr=lr)

    # 2 epochs per trial to keep search fast
    for epoch in range(2):
        trial_model.train()
        for (user_num, user_cat, cand_news_num, cand_news_cat,
             cand_bert, cand_topic, cand_entity,
             context_features, knowledge, _) in trial_loader:

            user_num         = user_num.to(device)
            user_cat         = {k: v.to(device) for k, v in user_cat.items()}
            cand_news_num    = cand_news_num.to(device)
            cand_news_cat    = cand_news_cat.to(device)
            cand_bert        = cand_bert.to(device)
            cand_topic       = cand_topic.to(device)
            cand_entity      = cand_entity.to(device)
            context_features = context_features.to(device)
            knowledge        = knowledge.to(device)

            B = user_num.shape[0]
            opt.zero_grad()
            scores = score_candidates(
                trial_model, user_num, user_cat,
                cand_news_num, cand_news_cat,
                cand_bert, cand_topic, cand_entity,
                knowledge, context_features, device
            )
            F.cross_entropy(
                scores,
                torch.zeros(B, dtype=torch.long, device=device)
            ).backward()
            opt.step()

    # Evaluate on val_loader (variable-length, batch_size=1)
    trial_model.eval()
    val_labels_all, val_scores_all, val_imp_ids = [], [], []
    MAX_VAL_IMPRESSIONS = 3000  # subsample for speed during tuning

    with torch.no_grad():
        for batch_idx, (user_num, user_cat, cand_news_num, cand_news_cat,
             cand_bert, cand_topic, cand_entity,
             context_batch, knowledge, labels_batch,
             impression_ids_batch) in enumerate(val_loader):

            if batch_idx >= MAX_VAL_IMPRESSIONS:
                break

            user_num      = user_num.to(device)
            user_cat      = {k: v.to(device) for k, v in user_cat.items()}
            cand_news_num = cand_news_num.to(device)
            cand_news_cat = cand_news_cat.to(device)
            cand_bert     = cand_bert.to(device)
            cand_topic    = cand_topic.to(device)
            cand_entity   = cand_entity.to(device)
            context_batch = context_batch.to(device)
            knowledge     = knowledge.to(device)

            scores = score_candidates(
                trial_model, user_num, user_cat,
                cand_news_num, cand_news_cat,
                cand_bert, cand_topic, cand_entity,
                context_batch, knowledge, device
            )
            val_labels_all.extend(labels_batch.cpu().numpy().tolist())
            val_scores_all.extend(
                torch.softmax(scores, dim=1).cpu().numpy().tolist()
            )
            val_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

    _, _, val_ndcg5, _ = calculate_ranking_metrics(
        val_imp_ids, val_labels_all, val_scores_all
    )
    return val_ndcg5


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print(f"\nBest val nDCG@5 : {study.best_value:.4f}")
print(f"Best params     : {study.best_params}")


Best val nDCG@5 : 0.2955
Best params     : {'lr': 0.0009911881629705145, 'embed_dim': 64, 'batch_size': 32}


We conclude that the best hyperparameters below will be applied to the final model in Section xx.
* `lr` = 1e-3
* `embed_dim` = 64
* `batch_size` = 32

##5.4 Training Loop

In [ ]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# The positive candidate is always at index 0 in every group.
# Listwise CE target is therefore always 0 for every impression in the batch.
# We create it once here and reuse it (resizing per batch below).
TARGET_IDX = 0

# Lists to store metrics for plotting
train_losses_hist, val_losses_hist = [], []
train_aucs_hist, val_aucs_hist = [], []
val_mrrs_hist, val_maps_hist, val_ndcg5s_hist, val_ndcg10s_hist = [], [], [], []

for epoch in range(num_epochs):
    model.train()
    total_train_loss_epoch = 0.0
    total_train_samples_epoch = 0
    running_loss = 0.0
    train_all_labels  = []   # list of [5]-length label arrays, one per impression
    train_all_scores  = []   # list of [5]-length score arrays
    train_imp_ids     = []

    for i, (user_num, user_cat, cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            context_features, knowledge, impression_ids_batch) in enumerate(train_loader): # Added context_features

        # Move to device
        user_num      = user_num.to(device)
        user_cat      = {k: v.to(device) for k, v in user_cat.items()}
        cand_news_num = cand_news_num.to(device)
        cand_news_cat = cand_news_cat.to(device)
        cand_bert     = cand_bert.to(device)
        cand_topic    = cand_topic.to(device)
        cand_entity   = cand_entity.to(device)
        context_features = context_features.to(device) # Moved context_features to device
        knowledge     = knowledge.to(device)

        B = user_num.shape[0]

        optimizer.zero_grad()

        # Forward: scores [B, 5]
        scores = score_candidates(
            model, user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            knowledge, context_features,
            device
        )

        # Listwise cross-entropy loss
        # Target = 0 for all impressions (positive is always index 0)
        targets = torch.zeros(B, dtype=torch.long, device=device)
        loss    = F.cross_entropy(scores, targets)

        loss.backward()
        optimizer.step()

        total_train_loss_epoch += loss.item() * B # Accumulate total loss for the epoch
        total_train_samples_epoch += B
        running_loss += loss.item()

        # Collect for metrics — labels are [1,0,0,0,0] for each impression
        softmax_scores = torch.softmax(scores, dim=1).cpu().detach().numpy()  # [B, 5]
        labels_fixed   = [[1, 0, 0, 0, 0]] * B

        train_all_labels.extend(labels_fixed)
        train_all_scores.extend(softmax_scores.tolist())
        train_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

        if i % 100 == 99:
            print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss / 100:.4f}")
            running_loss = 0.0

    # Calculate epoch average training loss
    avg_train_loss = total_train_loss_epoch / total_train_samples_epoch
    train_losses_hist.append(avg_train_loss)

    # Training AUC — use score of the positive (index 0) vs max of negatives
    # roc_auc_score needs flat binary labels + scores
    flat_train_labels = [label for sublist in train_all_labels for label in sublist]
    flat_train_scores = [score for sublist in train_all_scores for score in sublist]
    if len(np.unique(flat_train_labels)) > 1:
        train_auc = roc_auc_score(flat_train_labels, flat_train_scores)
        print(f"Epoch {epoch+1} Train AUC: {train_auc:.4f}")
        train_aucs_hist.append(train_auc)
    else:
        print(f"Epoch {epoch+1} Train AUC: N/A")
        train_aucs_hist.append(0.0) # Store 0.0 if AUC cannot be calculated

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_loss      = 0.0
    val_all_labels  = []
    val_all_scores  = []
    val_imp_ids     = []

    with torch.no_grad():
        for (user_num, user_cat, cand_news_num, cand_news_cat,
             cand_bert, cand_topic, cand_entity,
             context_batch, knowledge, labels_batch, impression_ids_batch) in val_loader:

            user_num      = user_num.to(device)
            user_cat      = {k: v.to(device) for k, v in user_cat.items()}
            cand_news_num = cand_news_num.to(device)
            cand_news_cat = cand_news_cat.to(device)
            cand_bert     = cand_bert.to(device)
            cand_topic    = cand_topic.to(device)
            cand_entity   = cand_entity.to(device)
            knowledge     = knowledge.to(device)
            context_batch = context_batch.to(device)
            labels_batch  = labels_batch.to(device)

            B = user_num.shape[0] # B will be 1 for variable collate fn

            scores  = score_candidates(
                model, user_num, user_cat,
                cand_news_num, cand_news_cat,
                cand_bert, cand_topic, cand_entity,
                context_batch, # Correctly pass context_batch as context_features
                knowledge,     # Correctly pass knowledge as knowledge
                device
            )
            # BCE Loss
            loss    = F.binary_cross_entropy_with_logits(scores, labels_batch) # Use labels_batch
            val_loss += loss.item()

            softmax_scores = torch.softmax(scores, dim=1).cpu().numpy()
            # Labels should be taken from labels_batch, not fixed
            val_all_labels.extend(labels_batch.cpu().numpy().tolist())
            val_all_scores.extend(softmax_scores.tolist())
            val_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

    avg_val_loss = val_loss / len(val_loader)

    # Ranking metrics (group-level: one entry per impression)
    avg_mrr, avg_map, avg_ndcg_5, avg_ndcg_10 = calculate_ranking_metrics(
        val_imp_ids, val_all_labels, val_all_scores
    )

    flat_val_labels = [label for sublist in val_all_labels for label in sublist]
    flat_val_scores = [score for sublist in val_all_scores for score in sublist]
    if len(np.unique(flat_val_labels)) > 1:
        val_auc = roc_auc_score(flat_val_labels, flat_val_scores)
        print(f"Epoch {epoch+1} Val Loss: {avg_val_loss:.4f}, Val AUC: {val_auc:.4f}")
        val_aucs_hist.append(val_auc)
    else:
        print(f"Epoch {epoch+1} Val Loss: {avg_val_loss:.4f}, Val AUC: N/A")
        val_aucs_hist.append(0.0) # Store 0.0 if AUC cannot be calculated

    print(f"  MRR: {avg_mrr:.4f}  MAP: {avg_map:.4f}  "
          f"nDCG@5: {avg_ndcg_5:.4f}  nDCG@10: {avg_ndcg_10:.4f}")
    val_losses_hist.append(avg_val_loss)
    val_mrrs_hist.append(avg_mrr)
    val_maps_hist.append(avg_map)
    val_ndcg5s_hist.append(avg_ndcg_5)
    val_ndcg10s_hist.append(avg_ndcg_10)

print('Finished Training')

# Store baseline result for comparison in Section 7
baseline_result = {
    'config':     'full_model_baseline',
    'removed':    'none',
    'val_auc':    round(max(val_aucs_hist),    4),
    'val_mrr':    round(max(val_mrrs_hist),    4),
    'val_ndcg5':  round(max(val_ndcg5s_hist),  4),
    'val_ndcg10': round(max(val_ndcg10s_hist), 4),
    'best_epoch': val_ndcg5s_hist.index(max(val_ndcg5s_hist)) + 1,
}
print(f"\nBaseline stored: {baseline_result}")

Epoch 1, Batch 100, Loss: 1.5479
Epoch 1, Batch 200, Loss: 1.5208
Epoch 1, Batch 300, Loss: 1.4810
Epoch 1, Batch 400, Loss: 1.4677
Epoch 1, Batch 500, Loss: 1.4706
Epoch 1, Batch 600, Loss: 1.4725
Epoch 1, Batch 700, Loss: 1.4647
Epoch 1, Batch 800, Loss: 1.4553
Epoch 1, Batch 900, Loss: 1.4498
Epoch 1, Batch 1000, Loss: 1.4425
Epoch 1, Batch 1100, Loss: 1.4356
Epoch 1, Batch 1200, Loss: 1.4288
Epoch 1, Batch 1300, Loss: 1.4240
Epoch 1, Batch 1400, Loss: 1.4217
Epoch 1, Batch 1500, Loss: 1.4529
Epoch 1, Batch 1600, Loss: 1.4108
Epoch 1, Batch 1700, Loss: 1.4055
Epoch 1, Batch 1800, Loss: 1.4304
Epoch 1, Batch 1900, Loss: 1.4179
Epoch 1, Batch 2000, Loss: 1.4010
Epoch 1, Batch 2100, Loss: 1.3870
Epoch 1, Batch 2200, Loss: 1.3910
Epoch 1, Batch 2300, Loss: 1.4027
Epoch 1, Batch 2400, Loss: 1.4008
Epoch 1, Batch 2500, Loss: 1.4214
Epoch 1, Batch 2600, Loss: 1.3843
Epoch 1, Batch 2700, Loss: 1.3911
Epoch 1, Batch 2800, Loss: 1.3748
Epoch 1, Batch 2900, Loss: 1.4004
Epoch 1, Batch 3000, Lo

Findings:

* **Val AUC** (Global Classification) peaks very early at Epoch 2 (0.6415) and steadily declines thereafter.
* **nDCG** and **MRR** (Local Ranking) continue to improve until Epoch 4 (nDCG@5 hits 0.2903).

This indicates that as training progresses, the model becomes worse at distinguishing generally "good" articles from "bad" articles across the whole dataset (dropping AUC), but it becomes much better at comparing 5 specific articles **within a single user's feed** and putting the best one at the top (rising nDCG).

For the MIND recommender system, such ranking metrics (nDCG/MRR) prove that the model is performing optimally.

#6 Test Evaluation

In [ ]:
model.eval()
test_all_labels = []
test_all_scores = []
test_imp_ids    = []

with torch.no_grad():
    for (user_num, user_cat, cand_news_num, cand_news_cat,
         cand_bert, cand_topic, cand_entity,
         context_features, knowledge, labels_batch, impression_ids_batch) in test_loader: # Added context_features

        user_num      = user_num.to(device)
        user_cat      = {k: v.to(device) for k, v in user_cat.items()}
        cand_news_num = cand_news_num.to(device)
        cand_news_cat = cand_news_cat.to(device)
        cand_bert     = cand_bert.to(device)
        cand_topic    = cand_topic.to(device)
        cand_entity   = cand_entity.to(device)
        context_features = context_features.to(device) # Moved context_features to device
        knowledge     = knowledge.to(device)
        labels_batch  = labels_batch.to(device) # Move labels to device

        B      = user_num.shape[0] # B will be 1
        scores = score_candidates(
            model, user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            context_features, # Passing context features
            knowledge,
            device
        )

        softmax_scores = torch.softmax(scores, dim=1).cpu().numpy()
        # Labels should be taken from labels_batch, not fixed
        test_all_labels.extend(labels_batch.cpu().numpy().tolist())
        test_all_scores.extend(softmax_scores.tolist())
        test_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

# Ranking metrics
test_mrr, test_map, test_ndcg_5, test_ndcg_10 = calculate_ranking_metrics(
    test_imp_ids, test_all_labels, test_all_scores
)

# AUC
flat_test_labels = [label for sublist in test_all_labels for label in sublist]
flat_test_scores = [score for sublist in test_all_scores for score in sublist]
if len(np.unique(flat_test_labels)) > 1:
    test_auc = roc_auc_score(flat_test_labels, flat_test_scores)
    print(f"Test AUC: {test_auc:.4f}")

print(f"Test MRR: {test_mrr:.4f}  MAP: {test_map:.4f}  "
      f"nDCG@5: {test_ndcg_5:.4f}  nDCG@10: {test_ndcg_10:.4f}")

Test AUC: 0.6272
Test MRR: 0.2868  MAP: 0.2602  nDCG@5: 0.2585  nDCG@10: 0.3145


# 7 Feature Attribution Analysis

We now analyse the model to determine which specific features drive its reccommendations. This in conducted via Integrated Gradients. Our finding are summarised below, and we will explore further using the feature ablation tests in *Section 8*.

*Note*: This analysis is only be done on ***numerical*** features, and  the attribution percentages are only comparing numerical feature groups against each other, not the full model input.
* **News CTR Dominates:** The CTR of a news article is dominating the model behavior. The model is functioning almost entirely as a popularity-based ranker. It uses user features such as the Diversity Ratio and History Length to make minor adjustments, but the popularity of the article is dictating most of model behavior.


In [ ]:
!pip install captum

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 129.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have 

In [ ]:
import torch
from captum.attr import IntegratedGradients

model.eval()

num_samples_to_test = 50
processed_count = 0

# Trackers for our 4 continuous tensor groups
total_user_importance = torch.zeros(3).to(device)
total_context_importance = torch.zeros(4).to(device)
total_news_num_importance = torch.zeros(cand_news_num_batch.shape[-1]).to(device) # Dynamically size to your news num features
total_bert_importance = 0.0 # BERT is a massive 768/384 dim vector, we will track its overall mean magnitude

print(f"Calculating full continuous attributions across {num_samples_to_test} impressions...\n")

class FullCaptumWrapper(torch.nn.Module):
    def __init__(self, base_model, user_cat, cand_cat, topic, entity, knowledge):
        super().__init__()
        self.base_model = base_model
        self.user_cat = user_cat
        self.cand_cat = cand_cat
        self.topic = topic
        self.entity = entity
        self.knowledge = knowledge

    def forward(self, eval_user_num, eval_context, eval_cand_num, eval_bert):
        captum_batch = eval_user_num.shape[0]

        # Expand static categorical/embedding features
        cat_exp = {k: v.expand(captum_batch) for k, v in self.user_cat.items()}
        cand_cat_exp  = self.cand_cat.expand(captum_batch, -1)
        topic_exp     = self.topic.expand(captum_batch, -1, -1)
        entity_exp    = self.entity.expand(captum_batch, -1, -1)
        knowledge_exp = self.knowledge.expand(captum_batch, -1, -1)

        scores = score_candidates(
            self.base_model, eval_user_num, cat_exp,
            eval_cand_num, cand_cat_exp,
            eval_bert, topic_exp, entity_exp,
            eval_context, knowledge_exp, device
        )
        return scores

for i, batch in enumerate(val_loader):
    if processed_count >= num_samples_to_test:
        break

    (user_num_batch, user_cat_batch, cand_news_num_batch, cand_news_cat_batch,
     cand_bert_batch, cand_topic_batch, cand_entity_batch,
     context_batch, knowledge_batch, labels_batch, impression_ids_batch) = batch

    # Move to device
    user_num_test = user_num_batch.to(device)
    user_cat_test = {k: v.to(device) for k, v in user_cat_batch.items()}
    cand_news_num_test = cand_news_num_batch.to(device)
    cand_news_cat_test = cand_news_cat_batch.to(device)
    cand_bert_test = cand_bert_batch.to(device)
    cand_topic_test = cand_topic_batch.to(device)
    cand_entity_test = cand_entity_batch.to(device)
    context_test = context_batch.to(device)
    knowledge_test = knowledge_batch.to(device)
    labels_test = labels_batch.to(device)

    # Require gradients for ALL tested continuous inputs
    user_num_test.requires_grad_()
    context_test.requires_grad_()
    cand_news_num_test.requires_grad_()
    cand_bert_test.requires_grad_()

    target_idx = torch.argmax(labels_test[0]).item()
    if labels_test[0][target_idx] == 0:
        continue

    wrapper = FullCaptumWrapper(
        model, user_cat_test, cand_news_cat_test,
        cand_topic_test, cand_entity_test, knowledge_test
    ).to(device)

    ig = IntegratedGradients(wrapper)

    # Calculate Attributions for all 4 inputs
    attributions = ig.attribute(
        inputs=(user_num_test, context_test, cand_news_num_test, cand_bert_test),
        target=target_idx,
        n_steps=25
    )

    user_attr, context_attr, news_num_attr, bert_attr = attributions

    # Accumulate importance
    total_user_importance += user_attr.mean(dim=0).abs()
    total_context_importance += context_attr.mean(dim=(0, 1)).abs()
    # News features are shape [1, num_candidates, features]. Mean across batch and candidates.
    total_news_num_importance += news_num_attr.mean(dim=(0, 1)).abs()
    # BERT is massive, so we take the mean of the entire vector space to see its overall weight
    total_bert_importance += bert_attr.mean().abs().item()

    processed_count += 1
    if processed_count % 10 == 0:
        print(f"Processed {processed_count}/{num_samples_to_test} impressions...")

avg_user = total_user_importance / processed_count
avg_context = total_context_importance / processed_count
avg_news_num = total_news_num_importance / processed_count
avg_bert = total_bert_importance / processed_count

print("\nNote: We can only analyse NUMERICAL features. The breakdown does not take into account categorical features")

print("\n--- GLOBAL FEATURE IMPORTANCE (Averaged) ---")
print("USER:")
print(f"  History Length:      {avg_user[0].item():.6f}")
print(f"  Diversity Ratio:     {avg_user[1].item():.6f}")
print(f"  Concentration Ratio:  {avg_user[2].item():.6f}")

print("\nCONTEXT:")
print(f"  Hour (Sin):          {avg_context[0].item():.6f}")
print(f"  Hour (Cos):          {avg_context[1].item():.6f}")
print(f"  Habit Score:         {avg_context[2].item():.6f}")
print(f"  Recency Score:       {avg_context[2].item():.6f}")

print("\nNEWS METRICS:")
print(f"  News CTR:            {avg_news_num[0].item():.6f}")
# Add more prints here if your cand_news_num has more than 1 feature (e.g., avg_news_num[1])

print("\nTEXT EMBEDDINGS:")
print(f"  BERT (Overall Mean): {avg_bert:.6f}")

Calculating full continuous attributions across 50 impressions...

Processed 10/50 impressions...
Processed 20/50 impressions...
Processed 30/50 impressions...
Processed 40/50 impressions...
Processed 50/50 impressions...

Note: We can only analyse NUMERICAL features. The breakdown does not take into account categorical features

--- GLOBAL FEATURE IMPORTANCE (Averaged) ---
USER:
  History Length:      0.112992
  Diversity Ratio:     0.187412
  Concentration Ratio:  0.102344

CONTEXT:
  Hour (Sin):          0.003725
  Hour (Cos):          0.002212
  Habit Score:         0.001415
  Recency Score:       0.001415

NEWS METRICS:
  News CTR:            0.322064

TEXT EMBEDDINGS:
  BERT (Overall Mean): 0.000108


In [ ]:
from captum.attr import IntegratedGradients

model.eval()
importance_tracker = {}
processed = 0

# 1. Simplified Wrapper
def simple_forward(user_num, context, news_num, bert, topic, entity, knowledge, n_cat, u_cat_1, u_cat_2, u_cat_3, u_cat_4, u_cat_5, u_cat_6):
    u_cat_keys = list(user_category_mappers.keys())
    u_cat_dict = {
        u_cat_keys[0]: u_cat_1, u_cat_keys[1]: u_cat_2, u_cat_keys[2]: u_cat_3,
        u_cat_keys[3]: u_cat_4, u_cat_keys[4]: u_cat_5, u_cat_keys[5]: u_cat_6
    }
    # Return scores from our robust score_candidates function
    return score_candidates(model, user_num, u_cat_dict, news_num, n_cat, bert, topic, entity, context, knowledge, device)

ig = IntegratedGradients(simple_forward)

for i, batch in enumerate(val_loader):
    if processed >= 20: break # Small sample to verify

    (u_num, u_cat, n_num, n_cat, n_bert, n_top, n_ent, ctx, knw, lbl, _) = [
        b.to(device) if isinstance(b, torch.Tensor) else b for b in batch
    ]
    u_cat_list = [v.to(device).float() for v in u_cat.values()] # Float for IG baseline math

    target_idx = torch.argmax(lbl[0]).item()

    # Measure attributions for ALL continuous and categorical-index inputs
    attrs = ig.attribute(
        inputs=(u_num, ctx, n_num, n_bert, n_top, n_ent, knw),
        additional_forward_args=(n_cat, *u_cat_list),
        target=target_idx,
        n_steps=10
    )

    # Map the results
    labels = ["User Numerical", "Context", "News Numerical", "BERT", "Topics", "Entities", "Knowledge"]
    for idx, label in enumerate(labels):
        importance_tracker[label] = importance_tracker.get(label, 0) + attrs[idx].abs().mean().item()

    processed += 1

# Normalize
total = sum(importance_tracker.values())
print("\n--- FEATURE INFLUENCE BREAKDOWN (excl. categorical features) ---")
for label, val in sorted(importance_tracker.items(), key=lambda x: x[1], reverse=True):
    print(f"{label:25}: {(val/total)*100:>6.2f}%")


--- FEATURE INFLUENCE BREAKDOWN (excl. categorical features) ---
News Numerical           :  71.77%
User Numerical           :  26.11%
Knowledge                :   1.29%
Context                  :   0.33%
Entities                 :   0.25%
BERT                     :   0.20%
Topics                   :   0.06%


# 8 Feature Ablation Tests


## 8.1 General Feature Blocks Ablation
Each run below trains a fresh model with one feature group removed.

Primary comparison metric: val nDCG@5 (best epoch across 5 epochs).
Baseline result is prepended from the baseline_result dict stored at
the end of Section 5.4.

Ablation configs:
  1. **no_context**     — remove hour_sin, hour_cos, habit_score, recency_score
  2. **no_knowledge**   — remove knowledge match score
  3. **no_entity**      — remove entity embeddings
  4. **no_bert**        — remove BERT embeddings
  5. **no_news_emb**    — remove BERT + topics + entity together
  6. **no_user**        — remove all user features (numerical + categorical)

In [ ]:
# ── Section 7.1 — Ablation model + training function ─────────────────────────

class NewsMLP_Ablation(nn.Module):
    def __init__(
        self,
        user_numerical_dense_dim,
        user_categorical_vocab_sizes,
        news_dense_dim,
        news_categorical_vocab_size,
        bert_dim          = 768,
        topic_dim         = 50,
        entity_dim        = 100,
        context_dense_dim = 4,
        hidden_dims       = [128, 64, 32],
        embed_dim         = 128,
        use_context       = True,
        use_knowledge     = True,
        use_entity        = True,
        use_bert          = True,
        use_topic         = True,
        use_user          = True,
    ):
        super().__init__()
        self.embed_dim     = embed_dim
        self.use_context   = use_context
        self.use_knowledge = use_knowledge
        self.use_entity    = use_entity
        self.use_bert      = use_bert
        self.use_topic     = use_topic
        self.use_user      = use_user
        self.relu          = nn.ReLU()
        self.user_categorical_vocab_sizes = user_categorical_vocab_sizes

        # ── User branch ───────────────────────────────────────────────────────
        if use_user:
            self.user_numerical_dense_proj = nn.Linear(user_numerical_dense_dim, embed_dim)
            self.user_categorical_embeddings = nn.ModuleDict({
                col: nn.Embedding(vocab_size + 1, embed_dim)
                for col, vocab_size in user_categorical_vocab_sizes.items()
            })

        # ── News branch (always active) ───────────────────────────────────────
        self.news_dense_proj         = nn.Linear(news_dense_dim, embed_dim)
        self.news_category_embedding = nn.Embedding(news_categorical_vocab_size + 1, embed_dim)

        if use_bert:
            self.bert_proj  = nn.Linear(bert_dim, embed_dim)
        if use_topic:
            self.topic_proj = nn.Linear(topic_dim, embed_dim)
        if use_entity:
            self.entity_proj = nn.Linear(entity_dim, embed_dim)

        # ── Interaction branch ────────────────────────────────────────────────
        if use_knowledge:
            self.knowledge_proj = nn.Linear(1, embed_dim)

        # ── Context branch ────────────────────────────────────────────────────
        if use_context:
            self.context_proj = nn.Linear(context_dense_dim, embed_dim)

        # ── Compute fc1 input dim from active branches ────────────────────────
        n = 2  # news_num + news_cat always present
        if use_user:
            n += 1 + len(user_categorical_vocab_sizes)
        if use_bert:
            n += 1
        if use_topic:
            n += 1
        if use_entity:
            n += 1
        if use_knowledge:
            n += 1
        if use_context:
            n += 1

        self.fc1    = nn.Linear(embed_dim * n, hidden_dims[0])
        self.fc2    = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.fc3    = nn.Linear(hidden_dims[1], hidden_dims[2])
        self.output = nn.Linear(hidden_dims[2], 1)

    def forward(
        self,
        user_numerical_dense,
        user_categorical_input,
        news_numerical_dense,
        news_category_indices,
        bert_embed,
        topic_embed,
        entity_embed,
        knowledge_score,
        context_dense,
    ):
        parts = []

        if self.use_user:
            user_num_vec = self.relu(self.user_numerical_dense_proj(user_numerical_dense))
            user_cat_vecs = [
                self.user_categorical_embeddings[col](indices.long())
                for col, indices in user_categorical_input.items()
            ]
            parts += [user_num_vec] + user_cat_vecs

        # News — always included
        parts.append(self.relu(self.news_dense_proj(news_numerical_dense)))
        parts.append(self.relu(self.news_category_embedding(news_category_indices.long())))

        if self.use_bert:
            parts.append(self.relu(self.bert_proj(bert_embed)))
        if self.use_topic:
            parts.append(self.relu(self.topic_proj(topic_embed)))
        if self.use_entity:
            parts.append(self.relu(self.entity_proj(entity_embed)))
        if self.use_knowledge:
            parts.append(self.relu(self.knowledge_proj(knowledge_score)))
        if self.use_context:
            parts.append(self.relu(self.context_proj(context_dense)))

        x = torch.cat(parts, dim=1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.output(x).squeeze(1)


def score_candidates_ablation(model, user_num, user_cat,
                               cand_news_num, cand_news_cat,
                               cand_bert, cand_topic, cand_entity,
                               context, knowledge, device):
    B, num_cands = cand_news_num.shape[0], cand_news_num.shape[1]

    user_num_exp = user_num.unsqueeze(1).expand(-1, num_cands, -1).reshape(B * num_cands, -1)
    user_cat_exp = {
        col: v.unsqueeze(1).expand(-1, num_cands).reshape(B * num_cands)
        for col, v in user_cat.items()
    }

    # Always pass all tensors — forward() ignores what it doesn't need
    scores_flat = model(
        user_num_exp,
        user_cat_exp,
        cand_news_num.reshape(B * num_cands, -1),
        cand_news_cat.reshape(B * num_cands),
        cand_bert.reshape(B * num_cands, -1),
        cand_topic.reshape(B * num_cands, -1),
        cand_entity.reshape(B * num_cands, -1),
        knowledge.reshape(B * num_cands, -1),
        context.reshape(B * num_cands, -1),
    )
    return scores_flat.reshape(B, num_cands)


def run_ablation(config_name, flags, device, num_epochs=5):
    print(f"\n{'='*60}")
    print(f"Ablation: {config_name}  |  flags: {flags}")
    print(f"{'='*60}")

    abl_model = NewsMLP_Ablation(
        user_numerical_dense_dim     = user_numerical_dense_dim,
        user_categorical_vocab_sizes = user_vocab_sizes,
        news_dense_dim               = news_dense_dim,
        news_categorical_vocab_size  = news_categorical_vocab_size,
        bert_dim                     = bert_embeddings_tensor.shape[1],
        topic_dim                    = topics_tensor.shape[1],
        entity_dim                   = entity_embed_tensor.shape[1],
        context_dense_dim            = 4,
        hidden_dims                  = [128, 64, 32],
        embed_dim                    = 128,
        **flags,
    ).to(device)

    # Sanity check — confirm flags are stored correctly
    print(f"  Flags confirmed on model: "
          f"use_context={abl_model.use_context}, "
          f"use_knowledge={abl_model.use_knowledge}, "
          f"use_bert={abl_model.use_bert}, "
          f"use_topic={abl_model.use_topic}, "
          f"use_entity={abl_model.use_entity}, "
          f"use_user={abl_model.use_user}")

    optimizer = optim.Adam(abl_model.parameters(), lr=1e-4)

    best_ndcg5  = 0.0
    best_auc    = 0.0
    best_mrr    = 0.0
    best_map    = 0.0
    best_ndcg10 = 0.0
    best_epoch  = 0

    for epoch in range(num_epochs):

        # ── Train ─────────────────────────────────────────────────────────────
        abl_model.train()

        for (user_num, user_cat, cand_news_num, cand_news_cat,
             cand_bert, cand_topic, cand_entity,
             swapped_knowledge, swapped_context,
             impression_ids_batch) in train_loader:

            user_num         = user_num.to(device)
            user_cat         = {k: v.to(device) for k, v in user_cat.items()}
            cand_news_num    = cand_news_num.to(device)
            cand_news_cat    = cand_news_cat.to(device)
            cand_bert        = cand_bert.to(device)
            cand_topic       = cand_topic.to(device)
            cand_entity      = cand_entity.to(device)

            # Untangle the tensors and move to device
            actual_context   = swapped_context.to(device)
            actual_knowledge = swapped_knowledge.to(device)

            B = user_num.shape[0]
            optimizer.zero_grad()

            scores = score_candidates_ablation(
                abl_model, user_num, user_cat,
                cand_news_num, cand_news_cat,
                cand_bert, cand_topic, cand_entity,
                actual_context, actual_knowledge, device # Passed in the correct order
            )
            loss = F.cross_entropy(
                scores,
                torch.zeros(B, dtype=torch.long, device=device)
            )
            loss.backward()
            optimizer.step()

        # ── Validate ──────────────────────────────────────────────────────────
        abl_model.eval()
        val_labels_all, val_scores_all, val_imp_ids = [], [], []

        with torch.no_grad():
            for (user_num, user_cat, cand_news_num, cand_news_cat,
                 cand_bert, cand_topic, cand_entity,
                 context_batch, knowledge,
                 labels_batch, impression_ids_batch) in val_loader:

                user_num      = user_num.to(device)
                user_cat      = {k: v.to(device) for k, v in user_cat.items()}
                cand_news_num = cand_news_num.to(device)
                cand_news_cat = cand_news_cat.to(device)
                cand_bert     = cand_bert.to(device)
                cand_topic    = cand_topic.to(device)
                cand_entity   = cand_entity.to(device)
                context_batch = context_batch.to(device)
                knowledge     = knowledge.to(device)

                scores = score_candidates_ablation(
                    abl_model, user_num, user_cat,
                    cand_news_num, cand_news_cat,
                    cand_bert, cand_topic, cand_entity,
                    context_batch, knowledge, device
                )

                val_labels_all.extend(labels_batch.cpu().numpy().tolist())
                val_scores_all.extend(
                    torch.softmax(scores, dim=1).cpu().numpy().tolist()
                )
                val_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

        flat_labels = [l for row in val_labels_all for l in row]
        flat_scores = [s for row in val_scores_all for s in row]
        val_auc = roc_auc_score(flat_labels, flat_scores) \
                  if len(np.unique(flat_labels)) > 1 else 0.0

        val_mrr, val_map, val_ndcg5, val_ndcg10 = calculate_ranking_metrics(
            val_imp_ids, val_labels_all, val_scores_all
        )

        print(f"  Epoch {epoch+1} | AUC: {val_auc:.4f}  "
              f"MRR: {val_mrr:.4f}  nDCG@5: {val_ndcg5:.4f}  nDCG@10: {val_ndcg10:.4f}")

        if val_ndcg5 > best_ndcg5:
            best_ndcg5  = val_ndcg5
            best_auc    = val_auc
            best_mrr    = val_mrr
            best_map    = val_map
            best_ndcg10 = val_ndcg10
            best_epoch  = epoch + 1

    print(f"  → Best epoch {best_epoch} | nDCG@5: {best_ndcg5:.4f}")

    return {
        'config':     config_name,
        'best_epoch': best_epoch,
        'val_auc':    round(best_auc,    4),
        'val_mrr':    round(best_mrr,    4),
        'val_map':    round(best_map,    4),
        'val_ndcg5':  round(best_ndcg5,  4),
        'val_ndcg10': round(best_ndcg10, 4),
    }

In [ ]:
# ── Section 7.2 — Run all ablations ──────────────────────────────────────────

# Start results list with the baseline stored at end of Section 5.4
ablation_results = [baseline_result]

# Define the 6 ablation configs as (name, flags_dict)
# All flags default to True — we only set the ones being turned OFF
ablation_configs = [
    ('no_context',   dict(use_context=False)),
    ('no_knowledge', dict(use_knowledge=False)),
    ('no_entity',    dict(use_entity=False)),
    ('no_bert',      dict(use_bert=False)),
    ('no_news_emb',  dict(use_bert=False, use_topic=False, use_entity=False)),
    ('no_user',      dict(use_user=False)),
]

for config_name, flags in ablation_configs:
    result = run_ablation(config_name, flags, device, num_epochs=5)
    ablation_results.append(result)

# ── Section 7.3 — Results Table ───────────────────────────────────────────────

results_df = pd.DataFrame(ablation_results).sort_values('val_ndcg5', ascending=False)

print("\n" + "=" * 75)
print("FEATURE ABLATION RESULTS — sorted by val nDCG@5 (best epoch)")
print("=" * 75)
print(results_df[['config', 'best_epoch', 'val_auc',
                   'val_mrr', 'val_map', 'val_ndcg5', 'val_ndcg10']].to_string(index=False))
print("=" * 75)

# Delta vs baseline for easy reading
baseline_ndcg5 = baseline_result['val_ndcg5']
print("\nDelta vs baseline (val nDCG@5):")
for _, row in results_df.iterrows():
    delta = row['val_ndcg5'] - baseline_ndcg5
    sign  = '+' if delta >= 0 else ''
    print(f"  {row['config']:<20} {sign}{delta:.4f}")


Ablation: no_context  |  flags: {'use_context': False}
  Flags confirmed on model: use_context=False, use_knowledge=True, use_bert=True, use_topic=True, use_entity=True, use_user=True
  Epoch 1 | AUC: 0.6606  MRR: 0.2895  nDCG@5: 0.2656  nDCG@10: 0.3226
  Epoch 2 | AUC: 0.6588  MRR: 0.3030  nDCG@5: 0.2786  nDCG@10: 0.3363
  Epoch 3 | AUC: 0.6611  MRR: 0.3073  nDCG@5: 0.2827  nDCG@10: 0.3410
  Epoch 4 | AUC: 0.6581  MRR: 0.3107  nDCG@5: 0.2861  nDCG@10: 0.3444
  Epoch 5 | AUC: 0.6555  MRR: 0.3120  nDCG@5: 0.2876  nDCG@10: 0.3458
  → Best epoch 5 | nDCG@5: 0.2876

Ablation: no_knowledge  |  flags: {'use_knowledge': False}
  Flags confirmed on model: use_context=True, use_knowledge=False, use_bert=True, use_topic=True, use_entity=True, use_user=True
  Epoch 1 | AUC: 0.6462  MRR: 0.2874  nDCG@5: 0.2628  nDCG@10: 0.3212
  Epoch 2 | AUC: 0.6435  MRR: 0.3096  nDCG@5: 0.2821  nDCG@10: 0.3402
  Epoch 3 | AUC: 0.6527  MRR: 0.3141  nDCG@5: 0.2875  nDCG@10: 0.3453
  Epoch 4 | AUC: 0.6461  MRR: 0.

From the results, we see that the **interaction** feature (knowledge match score between user's reading history and candidate article), caused the AUC metric to decrease while the ndcg@10 metric increased. This concludes that not only is the interaction feature weak, but possibly introduces noise for ranking as well. Hence, we will omit it from the final model.

## 8.2 Ablation Test on News **CTR** Feature

From the Feature Attribution Analysis run in Section 8, we see that `ctr_norm` (news CTR) dominates model behavior and that the model has effectively learned to be a popularity ranker with minor user-preference adjustments.

We run the ablation test below, zeroing out `ctr_norm` to test if the other features will be able to guide model behavior without news CTR.

In [ ]:
# ── Section 7.4 — CTR Ablation ────────────────────────────────────────────────
# Test whether the model is over-relying on CTR (news popularity).
# We zero out ctr_norm in the news_numerical tensor and retrain from scratch.
# Everything else is identical to the baseline.

print("Running CTR ablation — zeroing out ctr_norm in news_numerical_tensor...")

# Zero out CTR — news_numerical_tensor is shape [M, 1] (just ctr_norm)
# We replace it with zeros so the model cannot use popularity as a signal
news_numerical_tensor_no_ctr = torch.zeros_like(news_numerical_tensor)

# Rebuild train dataset with CTR zeroed out
train_dataset_no_ctr = NewsListwiseDataset(
    user_numerical_features        = train_user_numerical_tensor,
    user_categorical_features_dict = train_user_categorical_tensors,
    pos_news_ids                   = train_user_features['pos_id'],
    neg_news_ids                   = train_user_features['neg_ids'],
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor_no_ctr,  # <-- zeroed
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_scores               = train_knowledge_score,
    context_scalar                 = train_context_scalar,
    recency_scores                 = train_recency_score,
    impression_ids                 = train_user_features['impression_id'],
)

val_dataset_no_ctr = NewsListwiseDatasetVariable(
    user_numerical_features        = val_user_numerical_tensor,
    user_categorical_features_dict = val_user_categorical_tensors,
    interactions_df                = val_interactions,
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor_no_ctr,  # <-- zeroed
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_lookup               = val_knowledge_lookup,
    recency_lookup                 = val_recency_lookup,
    context_scalar_tensor          = val_context_scalar,
)

train_loader_no_ctr = DataLoader(train_dataset_no_ctr, batch_size=64,
                                  shuffle=True, collate_fn=listwise_collate_fn)
val_loader_no_ctr   = DataLoader(val_dataset_no_ctr, batch_size=1,
                                  shuffle=False, collate_fn=variable_collate_fn)

# Fresh model — same architecture as baseline
ctr_abl_model = NewsMLP(
    user_numerical_dense_dim     = user_numerical_dense_dim,
    user_categorical_vocab_sizes = user_vocab_sizes,
    news_dense_dim               = news_dense_dim,
    news_categorical_vocab_size  = news_categorical_vocab_size,
    bert_dim                     = bert_embeddings_tensor.shape[1],
    topic_dim                    = topics_tensor.shape[1],
    entity_dim                   = entity_embed_tensor.shape[1],
    context_dense_dim            = 4,
    hidden_dims                  = [128, 64, 32],
    embed_dim                    = 128,
).to(device)

ctr_optimizer = optim.Adam(ctr_abl_model.parameters(), lr=1e-4)
num_epochs_ctr = 5

best_ctr_ndcg5 = 0.0
best_ctr_result = {}

for epoch in range(num_epochs_ctr):
    ctr_abl_model.train()
    for (user_num, user_cat, cand_news_num, cand_news_cat,
         cand_bert, cand_topic, cand_entity,
         context_features, knowledge, _) in train_loader_no_ctr:

        user_num         = user_num.to(device)
        user_cat         = {k: v.to(device) for k, v in user_cat.items()}
        cand_news_num    = cand_news_num.to(device)
        cand_news_cat    = cand_news_cat.to(device)
        cand_bert        = cand_bert.to(device)
        cand_topic       = cand_topic.to(device)
        cand_entity      = cand_entity.to(device)
        context_features = context_features.to(device)
        knowledge        = knowledge.to(device)

        B = user_num.shape[0]
        ctr_optimizer.zero_grad()
        scores = score_candidates(
            ctr_abl_model, user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            knowledge, context_features, device
        )
        F.cross_entropy(scores, torch.zeros(B, dtype=torch.long, device=device)).backward()
        ctr_optimizer.step()

    # Validate
    ctr_abl_model.eval()
    val_labels_all, val_scores_all, val_imp_ids = [], [], []
    with torch.no_grad():
        for (user_num, user_cat, cand_news_num, cand_news_cat,
             cand_bert, cand_topic, cand_entity,
             context_batch, knowledge, labels_batch,
             impression_ids_batch) in val_loader_no_ctr:

            user_num      = user_num.to(device)
            user_cat      = {k: v.to(device) for k, v in user_cat.items()}
            cand_news_num = cand_news_num.to(device)
            cand_news_cat = cand_news_cat.to(device)
            cand_bert     = cand_bert.to(device)
            cand_topic    = cand_topic.to(device)
            cand_entity   = cand_entity.to(device)
            context_batch = context_batch.to(device)
            knowledge     = knowledge.to(device)

            scores = score_candidates(
                ctr_abl_model, user_num, user_cat,
                cand_news_num, cand_news_cat,
                cand_bert, cand_topic, cand_entity,
                context_batch, knowledge, device
            )
            val_labels_all.extend(labels_batch.cpu().numpy().tolist())
            val_scores_all.extend(torch.softmax(scores, dim=1).cpu().numpy().tolist())
            val_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

    flat_labels = [l for row in val_labels_all for l in row]
    flat_scores = [s for row in val_scores_all for s in row]
    val_auc   = roc_auc_score(flat_labels, flat_scores) if len(np.unique(flat_labels)) > 1 else 0.0
    _, _, val_ndcg5, val_ndcg10 = calculate_ranking_metrics(val_imp_ids, val_labels_all, val_scores_all)

    print(f"  Epoch {epoch+1} | AUC: {val_auc:.4f}  nDCG@5: {val_ndcg5:.4f}  nDCG@10: {val_ndcg10:.4f}")

    if val_ndcg5 > best_ctr_ndcg5:
        best_ctr_ndcg5 = val_ndcg5
        best_ctr_result = {'val_auc': round(val_auc, 4), 'val_ndcg5': round(val_ndcg5, 4), 'val_ndcg10': round(val_ndcg10, 4)}

print(f"\nBaseline nDCG@5 : {baseline_result['val_ndcg5']:.4f}")
print(f"No-CTR   nDCG@5 : {best_ctr_ndcg5:.4f}")
print(f"Delta           : {best_ctr_ndcg5 - baseline_result['val_ndcg5']:+.4f}")

Running CTR ablation — zeroing out ctr_norm in news_numerical_tensor...
  Epoch 1 | AUC: 0.7128  nDCG@5: 0.2853  nDCG@10: 0.3455
  Epoch 2 | AUC: 0.7231  nDCG@5: 0.2992  nDCG@10: 0.3598
  Epoch 3 | AUC: 0.7277  nDCG@5: 0.3044  nDCG@10: 0.3654
  Epoch 4 | AUC: 0.7268  nDCG@5: 0.3094  nDCG@10: 0.3684
  Epoch 5 | AUC: 0.7327  nDCG@5: 0.3145  nDCG@10: 0.3740

Baseline nDCG@5 : 0.2904
No-CTR   nDCG@5 : 0.3145
Delta           : +0.0241


Surprisingly, we see the best metric result generated for NDCG@5 (reaching 0.31) after this ablation test of dropping news ctr. This is a +0.0241 improvement in the NDCG@5 after dropping the news CTR feature. This is larger than any single feature group in the original ablation in Section 7.1. Therefore, we can conclude that **CTR was actively hurting ranking quality**.

Hence, we will drop news CTR from the final model.

# 9 Performance and Bias Analysis

## Category Bias Analysis
We evaluate if the model performs differently across content categories by comparing ranking metrics across the top, middle and bottom 5 categories by impressions against overall performance.



Extracting top and middle 5 categories and bottom 20 categories by number of impressions.

We observe the bottom 20 to elimate categories with too few impressions as they will be statistically insignificant

In [ ]:
# Combine all interaction data to count impressions per category
all_pos_news_ids = []

# For train_interactions, 'pos_id' is already a scalar
all_pos_news_ids.extend(train_interactions['pos_id'].tolist())

# For val_interactions and test_interactions, 'pos_ids' are lists, take the first element
all_pos_news_ids.extend([x[0] for x in val_interactions['pos_ids'].tolist()])
all_pos_news_ids.extend([x[0] for x in test_interactions['pos_ids'].tolist()])

# Map news_ids to categories (assuming news_id_to_category is already defined)
# news_id_to_category was created in cell xW1eC0kFiMRR
impression_categories = [news_id_to_category.get(nid, 'unknown') for nid in all_pos_news_ids]

# Count the occurrences of each category
category_impression_counts = pd.Series(impression_categories).value_counts()

# Exclude 'unknown' category if present
if 'unknown' in category_impression_counts:
    category_impression_counts = category_impression_counts.drop('unknown')

# Sort categories by impression count
sorted_category_impressions = category_impression_counts.sort_values(ascending=False)

# Extract top 5
top_5_categories = sorted_category_impressions.head(5)

# Extract bottom 20
bottom_20_categories = sorted_category_impressions.tail(20)

# Extract middle 5
num_total_categories = len(sorted_category_impressions)
if num_total_categories >= 15: # Ensure there are enough categories for top, middle, and bottom 5
    start_idx_middle = (num_total_categories // 2) - 2 # Center around the middle
    end_idx_middle = start_idx_middle + 5

    # Adjust if indices go out of bounds for smaller datasets
    if start_idx_middle < 0:
        start_idx_middle = 0
        end_idx_middle = min(5, num_total_categories)
    elif end_idx_middle > num_total_categories:
        end_idx_middle = num_total_categories
        start_idx_middle = max(0, num_total_categories - 5)

    middle_5_categories = sorted_category_impressions.iloc[start_idx_middle:end_idx_middle]
else:
    middle_5_categories = pd.Series([]) # Empty if not enough categories

print("Top 5 News Categories by Impressions:")
print(top_5_categories)
print("\nMiddle 5 News Categories by Impressions:")
print(middle_5_categories)
print("\nBottom 20 News Categories by Impressions:")
print(bottom_20_categories)

Top 5 News Categories by Impressions:
US_ColdCases_PoliceMisconduct    12029
US_SocialJustice                  9857
finance-companies                 8882
Deaths_MissingPersons             8238
lifestyle_general                 7987
Name: count, dtype: int64

Middle 5 News Categories by Impressions:
newsgoodnews                          1411
wellness                              1398
NCAA_PlayerSpotlight                  1382
US_CommunityDevelopment_FamilyLife    1312
news_general                          1292
Name: count, dtype: int64

Bottom 20 News Categories by Impressions:
2020Election_LegislativeReform    452
newsoffbeat                       317
MLB_Awards_Rankings               291
golf                              258
football_nfl_videos               236
lifestyle                         217
soccer                            210
science                           201
mma                               194
video_general                     178
lifestylefamily                   1

From the results, we choose these categories:


Top 5 categories:
- US_ColdCases_PoliceMisconduct
- US_SocialJustice  
- finance-companies
- Deaths_MissingPersons
- lifestyle_general

Middle 5 categories:
- newsgoodnews
- wellness
- NCAA_PlayerSpotlight
- US_CommunityDevelopment_FamilyLife
- news_general

Botttom 5 Categories:
- 2020Election_LegislativeReform
- newsoffbeat
- golf
- MLB_Awards_Rankings
- science

*Avoiding categories with too few impressions as they are less statistically significant

In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# ── Attach category labels to test impressions ──────────
# Each impression's positive article determines its category.
# We look up the category of pos_id from news_features.

# Build news_id → subclustered_category lookup
news_id_to_category = dict(zip(news_features['news_id'], news_features['subclustered_category']))

# Build impression_id → category from test_interactions
# The positive article's category represents the impression's category
test_interactions_reset = test_interactions.reset_index(drop=True)
impression_to_category = {}
for _, row in test_interactions_reset.iterrows():
    imp_id = row['impression_id']
    pos_id = row['pos_id'] if 'pos_id' in row else row['pos_ids'][0]
    impression_to_category[imp_id] = news_id_to_category.get(pos_id, 'unknown')

# ── Group test results by impression ──────────────
# test_all_labels, test_all_scores, test_imp_ids are already collected from section 6
# Each entry corresponds to one impression (variable-length candidate list)

top_categories = [
    'US_ColdCases_PoliceMisconduct',
    'US_SocialJustice',
    'finance-companies',
    'Deaths_MissingPersons',
    'lifestyle_general'
]

middle_categories = [
    'newsgoodnews',
    'wellness',
    'NCAA_PlayerSpotlight',
    'US_CommunityDevelopment_FamilyLife',
    'news_general'
]

bottom_categories = [
    '2020Election_LegislativeReform',
    'newsoffbeat',
    'golf',
    'MLB_Awards_Rankings',
    'science'
]

# ── Per-category metric calculation ────────────

def compute_metrics_for_impressions(labels_list, scores_list):
    """
    Given lists of per-impression label arrays and score arrays,
    compute MRR, MAP, AUC, nDCG@5, nDCG@10.
    """
    mrr_scores, map_scores, ndcg5_scores, ndcg10_scores = [], [], [], []
    flat_labels, flat_scores = [], []

    for labels, scores in zip(labels_list, scores_list):
        y_true  = np.array(labels)
        y_score = np.array(scores)

        flat_labels.extend(y_true.tolist())
        flat_scores.extend(y_score.tolist())

        if np.sum(y_true) == 0:
            continue

        mrr_scores.append(mrr_at_k(y_true, y_score))
        map_scores.append(average_precision_at_k(y_true, y_score))

        ranked      = sorted(zip(y_score, y_true), key=lambda x: x[0], reverse=True)
        rel_at_rank = [item[1] for item in ranked]
        ndcg5_scores.append(ndcg_at_k(rel_at_rank, 5))
        ndcg10_scores.append(ndcg_at_k(rel_at_rank, 10))

    # AUC needs both classes present
    auc = roc_auc_score(flat_labels, flat_scores) \
          if len(np.unique(flat_labels)) > 1 else float('nan')

    return {
        'n_impressions': len(labels_list),
        'MRR':     np.mean(mrr_scores)   if mrr_scores   else float('nan'),
        'MAP':     np.mean(map_scores)   if map_scores   else float('nan'),
        'AUC':     auc,
        'nDCG@5':  np.mean(ndcg5_scores) if ndcg5_scores else float('nan'),
        'nDCG@10': np.mean(ndcg10_scores)if ndcg10_scores else float('nan'),
    }

# ──  Bucket impressions by category and compute metrics ──────────────

# Overall baseline (all test impressions)
overall_metrics = compute_metrics_for_impressions(test_all_labels, test_all_scores)
overall_metrics['category'] = 'OVERALL'

results = [overall_metrics]

all_category_groups = [
    (top_categories, "---------- TOP 5 ----------"),
    (middle_categories, "---------- MIDDLE 5 ----------"),
    (bottom_categories, "---------- BOTTOM 5 ----------") # No separator after the last group
]

for i, (cat_group, separator_text) in enumerate(all_category_groups):
    if separator_text:
        results.append({
            'category': separator_text,
            'n_impressions': np.nan, 'MRR': np.nan, 'MAP': np.nan,
            'AUC': np.nan, 'nDCG@5': np.nan, 'nDCG@10': np.nan,
        })

    for category in cat_group:
        # Find indices where the impression belongs to this category
        cat_labels = []
        cat_scores = []
        for imp_id, labels, scores in zip(test_imp_ids, test_all_labels, test_all_scores):
            if impression_to_category.get(imp_id) == category:
                cat_labels.append(labels)
                cat_scores.append(scores)

        if len(cat_labels) == 0:
            print(f"Warning: No test impressions found for category '{category}'")
            results.append({
                'category': category, 'n_impressions': 0,
                'MRR': float('nan'), 'MAP': float('nan'),
                'AUC': float('nan'), 'nDCG@5': float('nan'), 'nDCG@10': float('nan'),
            })
            continue

        metrics = compute_metrics_for_impressions(cat_labels, cat_scores)
        metrics['category'] = category
        results.append(metrics)

# ── Display results ───────────────

results_df = pd.DataFrame(results, columns=[
    'category', 'n_impressions', 'MRR', 'MAP', 'AUC', 'nDCG@5', 'nDCG@10'
])
results_df = results_df.set_index('category')

# Replace NaN values in metric columns with empty strings for separator rows
metric_cols = ['n_impressions', 'MRR', 'MAP', 'AUC', 'nDCG@5', 'nDCG@10']
for col in metric_cols:
    results_df[col] = results_df[col].replace({np.nan: ''})

pd.set_option('display.float_format', '{:.4f}'.format)
print("\n===== Category Bias Analysis (Test Set) =====")
print(results_df.to_string())

# ── Bias summary: deviation from overall ─────────────

print("\n===== Deviation from Overall Performance =====")
overall_row = results_df.loc['OVERALL']
metric_cols = ['MRR', 'MAP', 'AUC', 'nDCG@5', 'nDCG@10']

# Print deviation for each group with separators
for i, (cat_group, separator_text) in enumerate(all_category_groups):
    if separator_text:
        print(f"\n{separator_text}")

    deviation_group_df = results_df.loc[cat_group, metric_cols].copy()
    for col in metric_cols:
        deviation_group_df[col] = pd.to_numeric(deviation_group_df[col], errors='coerce') - pd.to_numeric(overall_row[col], errors='coerce')
    print(deviation_group_df.to_string())



===== Category Bias Analysis (Test Set) =====
                                   n_impressions    MRR    MAP    AUC nDCG@5 nDCG@10
category                                                                            
OVERALL                               30270.0000 0.2868 0.2602 0.6272 0.2585  0.3145
---------- TOP 5 ----------                                                         
US_ColdCases_PoliceMisconduct          1357.0000 0.2570 0.2397 0.5948 0.2783  0.3325
US_SocialJustice                       1186.0000 0.3134 0.2888 0.6304 0.2968  0.3402
finance-companies                       918.0000 0.2882 0.2608 0.6773 0.2862  0.3409
Deaths_MissingPersons                   671.0000 0.3370 0.2859 0.6528 0.3043  0.3586
lifestyle_general                      1264.0000 0.1812 0.1570 0.5524 0.1403  0.1804
---------- MIDDLE 5 ----------                                                      
newsgoodnews                            320.0000 0.1807 0.1534 0.5341 0.1195  0.1933
wellness          

We cannot conclude there is category bias caused by dominating categories (by number of impressions) due to the mixed results of the analyses. The top, middle or bottom 5 categories do not cleanly map to better or worse performance.

**However, we can conclude that well-defined, specific categories perform better than broad, generic categories, introducing moderate categorical bias.**

Despite the subclustering of categories done during preprocessing, there still remains some categories that could not be clustered well enough (i.e. the subclusters contain broad topics), possibly due to the underlying data being heterogeneous. We also could not define too many categories as we already have 95 and subclustering further could introduce sparsity.


**Thus, we can conclude that further refinement of category features is unlikely to yield significant model improvements due to the abovementioned limitations and risk.**

# 10 Finalised Models

After the tests, the following steps will be applied to derive our final iteration of Model 2.

* Finalised hyperparameters
> * 'lr': 1e-3
> * 'embed_dim': 64,
> * 'batch_size': 32
* Drop interaction feature, `knowledge_match_score`
* Adjustment to news feature, `ctr_norm` for cold start users
> * Weighted `ctr_norm`, model to rely on news popularity only if user has a sparse reading history
> * Drop `ctr_norm` completely, model to rely con user profile only





## 10.1 Weighted CTR Model

In [ ]:
# ── Final Model: Step 1 — Weighted CTR Dataset Classes & Data Prep ────────────
# We define new dataset classes here so baseline classes in Section 3 are
# completely untouched. The only difference is __getitem__ multiplies
# ctr_norm by the user's cold-start weight before returning news_numerical.

class NewsListwiseDatasetWeighted(Dataset):
    """
    Identical to NewsListwiseDataset but applies per-user CTR weight
    to news_numerical features inside __getitem__.
    Cold-start users (history_length=0) → weight=1.0 (full CTR)
    Active users (history_length>=20)   → weight≈0.002 (negligible CTR)
    """
    def __init__(
        self,
        user_numerical_features,
        user_categorical_features_dict,
        pos_news_ids,
        neg_news_ids,
        news_id_to_idx_map,
        all_news_numerical,
        all_news_categorical,
        all_bert_embeddings,
        all_topic_embeddings,
        all_entity_embeddings,
        knowledge_scores,
        context_scalar,
        recency_scores,
        impression_ids,
        ctr_weights,               # NEW: [N, 1] per-user CTR weight tensor
    ):
        self.user_numerical   = user_numerical_features
        self.user_categorical = user_categorical_features_dict
        self.pos_news_ids     = pos_news_ids.reset_index(drop=True)
        self.neg_news_ids     = neg_news_ids.reset_index(drop=True)
        self.news_id_to_idx   = news_id_to_idx_map
        self.all_news_num     = all_news_numerical
        self.all_news_cat     = all_news_categorical
        self.all_bert         = all_bert_embeddings
        self.all_topic        = all_topic_embeddings
        self.all_entity       = all_entity_embeddings
        self.knowledge_score  = knowledge_scores
        self.context_scalar   = context_scalar
        self.recency_scores   = recency_scores
        self.impression_ids   = impression_ids.reset_index(drop=True)
        self.ctr_weights      = ctr_weights    # NEW

    def __len__(self):
        return len(self.pos_news_ids)

    def _get_news_features(self, news_id, ctr_weight):
        idx = self.news_id_to_idx.get(news_id, -1)
        if idx == -1:
            raise ValueError(f"News ID {news_id} not found in news_id_to_idx_map.")
        # Apply cold-start weight to ctr_norm before returning
        weighted_news_num = self.all_news_num[idx] * ctr_weight
        return (
            weighted_news_num,
            self.all_news_cat[idx],
            self.all_bert[idx],
            self.all_topic[idx],
            self.all_entity[idx],
        )

    def __getitem__(self, idx):
        user_num  = self.user_numerical[idx]
        user_cat  = {col: self.user_categorical[col][idx]
                     for col in self.user_categorical}

        ctr_weight = self.ctr_weights[idx]   # scalar weight for this user

        pos_feats = self._get_news_features(self.pos_news_ids[idx], ctr_weight)
        neg_ids   = self.neg_news_ids[idx]
        neg_feats = [self._get_news_features(nid, ctr_weight) for nid in neg_ids]

        all_candidates = [pos_feats] + neg_feats
        cand_news_num  = torch.stack([f[0] for f in all_candidates])
        cand_news_cat  = torch.stack([f[1] for f in all_candidates])
        cand_bert      = torch.stack([f[2] for f in all_candidates])
        cand_topic     = torch.stack([f[3] for f in all_candidates])
        cand_entity    = torch.stack([f[4] for f in all_candidates])
        knowledge      = self.knowledge_score[idx]
        recency        = self.recency_scores[idx]
        ctx_scalar     = self.context_scalar[idx]
        impression_id  = self.impression_ids[idx]

        ctx_scalar_expanded = ctx_scalar.unsqueeze(0).expand(5, -1)
        context = torch.cat([ctx_scalar_expanded, recency.unsqueeze(1)], dim=1)

        return (
            user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            knowledge, context,
            impression_id,
        )


class NewsListwiseDatasetVariableWeighted(Dataset):
    """
    Identical to NewsListwiseDatasetVariable but applies per-user CTR weight.
    """
    def __init__(
        self,
        user_numerical_features,
        user_categorical_features_dict,
        interactions_df,
        news_id_to_idx_map,
        all_news_numerical,
        all_news_categorical,
        all_bert_embeddings,
        all_topic_embeddings,
        all_entity_embeddings,
        knowledge_lookup,
        recency_lookup,
        context_scalar_tensor,
        ctr_weights,               # NEW: [N, 1] per-user CTR weight tensor
    ):
        self.user_numerical        = user_numerical_features
        self.user_categorical      = user_categorical_features_dict
        self.interactions          = interactions_df.reset_index(drop=True)
        self.news_id_to_idx        = news_id_to_idx_map
        self.all_news_num          = all_news_numerical
        self.all_news_cat          = all_news_categorical
        self.all_bert              = all_bert_embeddings
        self.all_topic             = all_topic_embeddings
        self.all_entity            = all_entity_embeddings
        self.knowledge_lookup      = knowledge_lookup
        self.recency_lookup        = recency_lookup
        self.context_scalar_tensor = context_scalar_tensor
        self.ctr_weights           = ctr_weights    # NEW

    def __len__(self):
        return len(self.interactions)

    def _get_news_features(self, news_id, ctr_weight):
        if isinstance(news_id, str) and news_id.startswith('N'):
            news_id = int(news_id[1:])
        idx = self.news_id_to_idx.get(news_id, -1)
        if idx == -1:
            raise ValueError(f"News ID {news_id} not found.")
        weighted_news_num = self.all_news_num[idx] * ctr_weight
        return (
            weighted_news_num,
            self.all_news_cat[idx],
            self.all_bert[idx],
            self.all_topic[idx],
            self.all_entity[idx],
        )

    def __getitem__(self, idx):
        row           = self.interactions.iloc[idx]
        impression_id = row['impression_id']
        candidates    = row['candidate_news']
        labels        = row['labels']

        user_num   = self.user_numerical[idx]
        user_cat   = {col: self.user_categorical[col][idx]
                      for col in self.user_categorical}
        ctr_weight = self.ctr_weights[idx]   # scalar weight for this user

        all_feats     = [self._get_news_features(nid, ctr_weight) for nid in candidates]
        cand_news_num = torch.stack([f[0] for f in all_feats])
        cand_news_cat = torch.stack([f[1] for f in all_feats])
        cand_bert     = torch.stack([f[2] for f in all_feats])
        cand_topic    = torch.stack([f[3] for f in all_feats])
        cand_entity   = torch.stack([f[4] for f in all_feats])

        ctx_scalar          = self.context_scalar_tensor[idx]
        ctx_scalar_expanded = ctx_scalar.unsqueeze(0).expand(len(candidates), -1)
        recency = torch.tensor(
            [self.recency_lookup.get((impression_id, nid), 0.0) for nid in candidates],
            dtype=torch.float32
        ).unsqueeze(1)
        context_features_combined = torch.cat([ctx_scalar_expanded, recency], dim=1)

        knowledge = torch.tensor(
            [self.knowledge_lookup.get((impression_id, nid), 0.0) for nid in candidates],
            dtype=torch.float32
        ).unsqueeze(1)

        labels_tensor = torch.tensor(labels, dtype=torch.float32)

        return (
            user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            context_features_combined,
            knowledge,
            labels_tensor,
            impression_id,
        )


# ── Build final weighted datasets ─────────────────────────────────────────────
final_train_dataset = NewsListwiseDatasetWeighted(
    user_numerical_features        = train_user_numerical_tensor,
    user_categorical_features_dict = train_user_categorical_tensors,
    pos_news_ids                   = train_user_features['pos_id'],
    neg_news_ids                   = train_user_features['neg_ids'],
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor,   # raw CTR — weighting happens inside
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_scores               = train_knowledge_score,
    context_scalar                 = train_context_scalar,
    recency_scores                 = train_recency_score,
    impression_ids                 = train_user_features['impression_id'],
    ctr_weights                    = train_ctr_weights,       # NEW
)

final_val_dataset = NewsListwiseDatasetVariableWeighted(
    user_numerical_features        = val_user_numerical_tensor,
    user_categorical_features_dict = val_user_categorical_tensors,
    interactions_df                = val_interactions,
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor,   # raw CTR
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_lookup               = val_knowledge_lookup,
    recency_lookup                 = val_recency_lookup,
    context_scalar_tensor          = val_context_scalar,
    ctr_weights                    = val_ctr_weights,         # NEW
)

final_test_dataset = NewsListwiseDatasetVariableWeighted(
    user_numerical_features        = test_user_numerical_tensor,
    user_categorical_features_dict = test_user_categorical_tensors,
    interactions_df                = test_interactions,
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor,   # raw CTR
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_lookup               = test_knowledge_lookup,
    recency_lookup                 = test_recency_lookup,
    context_scalar_tensor          = test_context_scalar,
    ctr_weights                    = test_ctr_weights,        # NEW
)

print(f"Final train impressions : {len(final_train_dataset)}")
print(f"Final val   impressions : {len(final_val_dataset)}")
print(f"Final test  impressions : {len(final_test_dataset)}")

Final train impressions : 141558
Final val   impressions : 31624
Final test  impressions : 30270


In [ ]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# ── Final Model: Step 2 — Initialise with tuned hyperparameters ──────────────
# Uses NewsMLP_Ablation with use_knowledge=False to drop knowledge_match_score
# Tuned hyperparameters: lr=1e-3, embed_dim=64, batch_size=32

final_learning_rate = 1e-3
final_batch_size    = 32
final_num_epochs    = 6

final_train_loader = DataLoader(final_train_dataset, batch_size=final_batch_size,
                                shuffle=True, collate_fn=listwise_collate_fn)
final_val_loader   = DataLoader(final_val_dataset,   batch_size=1,
                                shuffle=False, collate_fn=variable_collate_fn)
final_test_loader  = DataLoader(final_test_dataset,  batch_size=1,
                                shuffle=False, collate_fn=variable_collate_fn)

final_model = NewsMLP_Ablation(
    user_numerical_dense_dim     = user_numerical_dense_dim,
    user_categorical_vocab_sizes = user_vocab_sizes,
    news_dense_dim               = news_dense_dim,
    news_categorical_vocab_size  = news_categorical_vocab_size,
    bert_dim                     = bert_embeddings_tensor.shape[1],
    topic_dim                    = topics_tensor.shape[1],
    entity_dim                   = entity_embed_tensor.shape[1],
    context_dense_dim            = 4,
    hidden_dims                  = [128, 64, 32],
    embed_dim                    = 64,           # tuned
    use_knowledge                = False,        # dropped
    # all other flags default to True
).to(device)

final_optimizer = optim.Adam(final_model.parameters(), lr=final_learning_rate)
print(final_model)

NewsMLP_Ablation(
  (relu): ReLU()
  (user_numerical_dense_proj): Linear(in_features=3, out_features=64, bias=True)
  (user_categorical_embeddings): ModuleDict(
    (recent_fav_cat): Embedding(97, 64)
    (top_cat_1): Embedding(97, 64)
    (top_cat_2): Embedding(97, 64)
    (top_cat_3): Embedding(97, 64)
    (diversity_cluster): Embedding(5, 64)
    (interest_cluster): Embedding(5, 64)
  )
  (news_dense_proj): Linear(in_features=1, out_features=64, bias=True)
  (news_category_embedding): Embedding(96, 64)
  (bert_proj): Linear(in_features=768, out_features=64, bias=True)
  (topic_proj): Linear(in_features=50, out_features=64, bias=True)
  (entity_proj): Linear(in_features=100, out_features=64, bias=True)
  (context_proj): Linear(in_features=4, out_features=64, bias=True)
  (fc1): Linear(in_features=832, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (output): Linear(in_features=32, 

In [ ]:
from tqdm import tqdm

In [ ]:
# ── Final Model: Step 3 — Training loop ──────────────────────────────────────
final_train_losses, final_val_losses = [], []
final_val_aucs, final_val_mrrs = [], []
final_val_ndcg5s, final_val_ndcg10s = [], []
final_val_maps = []

for epoch in range(final_num_epochs):

    # ── Train ─────────────────────────────────────────────────────────────
    final_model.train()
    total_loss, total_samples = 0.0, 0

    for (user_num, user_cat, cand_news_num, cand_news_cat,
         cand_bert, cand_topic, cand_entity,
         swapped_knowledge, swapped_context,
         impression_ids_batch) in tqdm(final_train_loader,
                                       desc=f"Epoch {epoch+1}/{final_num_epochs} [train]",
                                       unit="batch"):

        user_num         = user_num.to(device)
        user_cat         = {k: v.to(device) for k, v in user_cat.items()}
        cand_news_num    = cand_news_num.to(device)
        cand_news_cat    = cand_news_cat.to(device)
        cand_bert        = cand_bert.to(device)
        cand_topic       = cand_topic.to(device)
        cand_entity      = cand_entity.to(device)
        actual_context   = swapped_context.to(device)
        actual_knowledge = swapped_knowledge.to(device)

        B = user_num.shape[0]
        final_optimizer.zero_grad()

        scores = score_candidates_ablation(
            final_model, user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            actual_context, actual_knowledge, device
        )
        loss = F.cross_entropy(scores, torch.zeros(B, dtype=torch.long, device=device))
        loss.backward()
        final_optimizer.step()

        total_loss    += loss.item() * B
        total_samples += B

    avg_train_loss = total_loss / total_samples
    final_train_losses.append(avg_train_loss)

    # ── Validate ──────────────────────────────────────────────────────────
    final_model.eval()
    val_labels_all, val_scores_all, val_imp_ids = [], [], []

    with torch.no_grad():
        for (user_num, user_cat, cand_news_num, cand_news_cat,
             cand_bert, cand_topic, cand_entity,
             context_batch, knowledge, labels_batch,
             impression_ids_batch) in final_val_loader:

            user_num      = user_num.to(device)
            user_cat      = {k: v.to(device) for k, v in user_cat.items()}
            cand_news_num = cand_news_num.to(device)
            cand_news_cat = cand_news_cat.to(device)
            cand_bert     = cand_bert.to(device)
            cand_topic    = cand_topic.to(device)
            cand_entity   = cand_entity.to(device)
            context_batch = context_batch.to(device)
            knowledge     = knowledge.to(device)

            scores = score_candidates_ablation(
                final_model, user_num, user_cat,
                cand_news_num, cand_news_cat,
                cand_bert, cand_topic, cand_entity,
                context_batch, knowledge, device
            )
            val_labels_all.extend(labels_batch.cpu().numpy().tolist())
            val_scores_all.extend(
                torch.softmax(scores, dim=1).cpu().numpy().tolist()
            )
            val_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

    flat_labels = [l for row in val_labels_all for l in row]
    flat_scores = [s for row in val_scores_all for s in row]
    val_auc = roc_auc_score(flat_labels, flat_scores) \
              if len(np.unique(flat_labels)) > 1 else 0.0
    val_mrr, val_map, val_ndcg5, val_ndcg10 = calculate_ranking_metrics(
        val_imp_ids, val_labels_all, val_scores_all
    )

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | "
          f"AUC: {val_auc:.4f}  MRR: {val_mrr:.4f}  MAP: {val_map:.4f}  "
          f"nDCG@5: {val_ndcg5:.4f}  nDCG@10: {val_ndcg10:.4f}")

    final_val_losses.append(val_auc)
    final_val_aucs.append(val_auc)
    final_val_mrrs.append(val_mrr)
    final_val_maps.append(val_map)
    final_val_ndcg5s.append(val_ndcg5)
    final_val_ndcg10s.append(val_ndcg10)

best_epoch_idx = final_val_ndcg5s.index(max(final_val_ndcg5s))
print("\nFinal Model Training Complete")
print(f"Best epoch: {best_epoch_idx + 1}  (by val nDCG@5)")
print(f"  nDCG@5  : {final_val_ndcg5s[best_epoch_idx]:.4f}")
print(f"  nDCG@10 : {final_val_ndcg10s[best_epoch_idx]:.4f}")
print(f"  AUC     : {final_val_aucs[best_epoch_idx]:.4f}")
print(f"  MRR     : {final_val_mrrs[best_epoch_idx]:.4f}")
print(f"  MAP     : {final_val_maps[best_epoch_idx]:.4f}")

Epoch 1/6 [train]: 100%|██████████| 4424/4424 [00:51<00:00, 85.64batch/s]


Epoch 1 | Train Loss: 1.4123 | AUC: 0.7255  MRR: 0.3320  MAP: 0.3062  nDCG@5: 0.3084  nDCG@10: 0.3677


Epoch 2/6 [train]: 100%|██████████| 4424/4424 [00:52<00:00, 84.65batch/s]


Epoch 2 | Train Loss: 1.3524 | AUC: 0.7307  MRR: 0.3387  MAP: 0.3116  nDCG@5: 0.3154  nDCG@10: 0.3753


Epoch 3/6 [train]: 100%|██████████| 4424/4424 [00:52<00:00, 84.99batch/s]


...

Epoch 4 | Train Loss: 1.3124 | AUC: 0.7333  MRR: 0.3449  MAP: 0.3183  nDCG@5: 0.3228  nDCG@10: 0.3820


Epoch 5/6 [train]: 100%|██████████| 4424/4424 [00:53<00:00, 82.20batch/s]


Epoch 5 | Train Loss: 1.2972 | AUC: 0.7344  MRR: 0.3469  MAP: 0.3200  nDCG@5: 0.3250  nDCG@10: 0.3845


Epoch 6/6 [train]: 100%|██████████| 4424/4424 [00:52<00:00, 84.50batch/s]


Epoch 6 | Train Loss: 1.2824 | AUC: 0.7321  MRR: 0.3472  MAP: 0.3197  nDCG@5: 0.3246  nDCG@10: 0.3839

Final Model Training Complete
Best epoch: 5  (by val nDCG@5)
  nDCG@5  : 0.3250
  nDCG@10 : 0.3845
  AUC     : 0.7344
  MRR     : 0.3469
  MAP     : 0.3200


In [ ]:
# ── Final Model: Step 4 — Test evaluation ────────────────────────────────────

final_model.eval()
test_labels_all, test_scores_all, test_imp_ids = [], [], []

with torch.no_grad():
    for (user_num, user_cat, cand_news_num, cand_news_cat,
         cand_bert, cand_topic, cand_entity,
         context_features, knowledge, labels_batch,
         impression_ids_batch) in final_test_loader:

        user_num         = user_num.to(device)
        user_cat         = {k: v.to(device) for k, v in user_cat.items()}
        cand_news_num    = cand_news_num.to(device)
        cand_news_cat    = cand_news_cat.to(device)
        cand_bert        = cand_bert.to(device)
        cand_topic       = cand_topic.to(device)
        cand_entity      = cand_entity.to(device)
        context_features = context_features.to(device)
        knowledge        = knowledge.to(device)

        scores = score_candidates_ablation(
            final_model, user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            context_features, knowledge, device
        )
        test_labels_all.extend(labels_batch.cpu().numpy().tolist())
        test_scores_all.extend(
            torch.softmax(scores, dim=1).cpu().numpy().tolist()
        )
        test_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

test_mrr, test_map, test_ndcg5, test_ndcg10 = calculate_ranking_metrics(
    test_imp_ids, test_labels_all, test_scores_all
)
flat_test_labels = [l for row in test_labels_all for l in row]
flat_test_scores = [s for row in test_scores_all for s in row]
test_auc = roc_auc_score(flat_test_labels, flat_test_scores) \
           if len(np.unique(flat_test_labels)) > 1 else 0.0

print("=" * 55)
print("FINAL MODEL — TEST RESULTS")
print("=" * 55)
print(f"  AUC     : {test_auc:.4f}")
print(f"  MRR     : {test_mrr:.4f}")
print(f"  MAP     : {test_map:.4f}")
print(f"  nDCG@5  : {test_ndcg5:.4f}")
print(f"  nDCG@10 : {test_ndcg10:.4f}")
print("=" * 55)

FINAL MODEL — TEST RESULTS
  AUC     : 0.7103
  MRR     : 0.3077
  MAP     : 0.2815
  nDCG@5  : 0.2803
  nDCG@10 : 0.3418


In [ ]:
!pip install captum

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 130.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have 

In [ ]:
# ── Final Model: Step 5 — Attribution analysis split by user history ──────────
# Verifies cold-start weighting is working:
# Cold-start users  → CTR attribution should be HIGH
# Active users      → CTR attribution should be near ZERO
from captum.attr import IntegratedGradients

final_model.eval()

COLD_START_THRESHOLD = 5    # history_length <= 5 → cold-start
ACTIVE_THRESHOLD     = 20   # history_length >= 20 → active user
MAX_SAMPLES_PER_GROUP = 30

def run_attribution_split(loader, user_features_df, ctr_weights_tensor, group_label):
    """Run attribution analysis on a filtered subset of users."""

    importance_tracker = {}
    processed = 0

    # Build a set of impression indices matching the group
    if group_label == 'cold_start':
        mask = user_features_df['history_length'] <= COLD_START_THRESHOLD
    else:
        mask = user_features_df['history_length'] >= ACTIVE_THRESHOLD
    valid_indices = set(np.where(mask.values)[0])

    def simple_forward_final(user_num, context, news_num, bert, topic, entity,
                              knowledge, n_cat, u_cat_1, u_cat_2, u_cat_3,
                              u_cat_4, u_cat_5, u_cat_6):
        u_cat_keys = list(user_category_mappers.keys())
        u_cat_dict = {
            u_cat_keys[0]: u_cat_1, u_cat_keys[1]: u_cat_2,
            u_cat_keys[2]: u_cat_3, u_cat_keys[3]: u_cat_4,
            u_cat_keys[4]: u_cat_5, u_cat_keys[5]: u_cat_6
        }
        return score_candidates_ablation(
            final_model, user_num, u_cat_dict,
            news_num, n_cat, bert, topic, entity,
            context, knowledge, device
        )

    ig = IntegratedGradients(simple_forward_final)

    for i, batch in enumerate(loader):
        if processed >= MAX_SAMPLES_PER_GROUP:
            break
        if i not in valid_indices:
            continue

        (u_num, u_cat, n_num, n_cat, n_bert, n_top, n_ent,
         ctx, knw, lbl, _) = [
            b.to(device) if isinstance(b, torch.Tensor) else b for b in batch
        ]
        u_cat_list = [v.to(device).float() for v in u_cat.values()]
        target_idx = torch.argmax(lbl[0]).item()

        attrs = ig.attribute(
            inputs=(u_num, ctx, n_num, n_bert, n_top, n_ent),
            additional_forward_args=(knw, n_cat, *u_cat_list),
            target=target_idx,
            n_steps=10
        )

        labels = ["User Numerical", "Context", "News CTR (weighted)",
                  "BERT", "Topics", "Entities"]
        for idx, label in enumerate(labels):
            importance_tracker[label] = (
                importance_tracker.get(label, 0) + attrs[idx].abs().mean().item()
            )
        processed += 1

    print(f"\n--- {group_label.upper()} USERS (n={processed}) ---")
    total = sum(importance_tracker.values())
    if total == 0:
        print("  No samples found for this group.")
        return
    for label, val in sorted(importance_tracker.items(),
                              key=lambda x: x[1], reverse=True):
        print(f"  {label:25}: {(val/total)*100:>6.2f}%")

# Run for cold-start users
run_attribution_split(
    final_val_loader, val_user_features,
    val_ctr_weights, 'cold_start'
)

# Run for active users
run_attribution_split(
    final_val_loader, val_user_features,
    val_ctr_weights, 'active'
)


--- COLD_START USERS (n=30) ---
  User Numerical           :  90.70%
  News CTR (weighted)      :   8.23%
  Context                  :   0.52%
  Entities                 :   0.26%
  BERT                     :   0.18%
  Topics                   :   0.11%

--- ACTIVE USERS (n=30) ---
  User Numerical           :  97.64%
  Context                  :   1.34%
  Entities                 :   0.42%
  BERT                     :   0.37%
  Topics                   :   0.19%
  News CTR (weighted)      :   0.05%


We see that after applying weights on the news CTR feature, news CTR feature is correctly weighted depending on user's reading history.
* For users with less than 5 articles in their history, 8% of the model behavior is guided by new CTR, **relative to other numerical features**.
* For users with a rich reading history of more than 20 articles in their history, 0.05% of the model behavior is guided by new CTR, **relative to other numerical features**.

## 10.2 No News CTR Feature Model

In [ ]:
# ── Final Model: Step 1 — Drop CTR from news_numerical_tensor ────────────────
# Zero out ctr_norm so the model cannot use popularity as a signal
news_numerical_tensor_final = torch.zeros_like(news_numerical_tensor)

# Rebuild all three datasets with CTR zeroed out
final_train_dataset = NewsListwiseDataset(
    user_numerical_features        = train_user_numerical_tensor,
    user_categorical_features_dict = train_user_categorical_tensors,
    pos_news_ids                   = train_user_features['pos_id'],
    neg_news_ids                   = train_user_features['neg_ids'],
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor_final,  # CTR zeroed
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_scores               = train_knowledge_score,
    context_scalar                 = train_context_scalar,
    recency_scores                 = train_recency_score,
    impression_ids                 = train_user_features['impression_id'],
)

final_val_dataset = NewsListwiseDatasetVariable(
    user_numerical_features        = val_user_numerical_tensor,
    user_categorical_features_dict = val_user_categorical_tensors,
    interactions_df                = val_interactions,
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor_final,  # CTR zeroed
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_lookup               = val_knowledge_lookup,
    recency_lookup                 = val_recency_lookup,
    context_scalar_tensor          = val_context_scalar,
)

final_test_dataset = NewsListwiseDatasetVariable(
    user_numerical_features        = test_user_numerical_tensor,
    user_categorical_features_dict = test_user_categorical_tensors,
    interactions_df                = test_interactions,
    news_id_to_idx_map             = news_id_to_idx_map,
    all_news_numerical             = news_numerical_tensor_final,  # CTR zeroed
    all_news_categorical           = news_categorical_tensor,
    all_bert_embeddings            = bert_embeddings_tensor,
    all_topic_embeddings           = topics_tensor,
    all_entity_embeddings          = entity_embed_tensor,
    knowledge_lookup               = test_knowledge_lookup,
    recency_lookup                 = test_recency_lookup,
    context_scalar_tensor          = test_context_scalar,
)

print(f"Final train impressions : {len(final_train_dataset)}")
print(f"Final val   impressions : {len(final_val_dataset)}")
print(f"Final test  impressions : {len(final_test_dataset)}")

Final train impressions : 141558
Final val   impressions : 31624
Final test  impressions : 30270


In [ ]:
# ── Final Model: Step 2 — Reinitialise model, optimizer, and dataloaders ─────

final_learning_rate = 1e-3
final_batch_size    = 32
final_num_epochs    = 6

final_train_loader = DataLoader(final_train_dataset, batch_size=final_batch_size,
                                shuffle=True, collate_fn=listwise_collate_fn)
final_val_loader   = DataLoader(final_val_dataset,   batch_size=1,
                                shuffle=False, collate_fn=variable_collate_fn)
final_test_loader  = DataLoader(final_test_dataset,  batch_size=1,
                                shuffle=False, collate_fn=variable_collate_fn)

final_model = NewsMLP_Ablation(
    user_numerical_dense_dim     = user_numerical_dense_dim,
    user_categorical_vocab_sizes = user_vocab_sizes,
    news_dense_dim               = news_dense_dim,
    news_categorical_vocab_size  = news_categorical_vocab_size,
    bert_dim                     = bert_embeddings_tensor.shape[1],
    topic_dim                    = topics_tensor.shape[1],
    entity_dim                   = entity_embed_tensor.shape[1],
    context_dense_dim            = 4,
    hidden_dims                  = [128, 64, 32],
    embed_dim                    = 64,
    use_knowledge                = False,
).to(device)

final_optimizer = optim.Adam(final_model.parameters(), lr=final_learning_rate)
print("Model reinitialised — no CTR version")
print(final_model)

Model reinitialised — no CTR version
NewsMLP_Ablation(
  (relu): ReLU()
  (user_numerical_dense_proj): Linear(in_features=3, out_features=64, bias=True)
  (user_categorical_embeddings): ModuleDict(
    (recent_fav_cat): Embedding(97, 64)
    (top_cat_1): Embedding(97, 64)
    (top_cat_2): Embedding(97, 64)
    (top_cat_3): Embedding(97, 64)
    (diversity_cluster): Embedding(5, 64)
    (interest_cluster): Embedding(5, 64)
  )
  (news_dense_proj): Linear(in_features=1, out_features=64, bias=True)
  (news_category_embedding): Embedding(96, 64)
  (bert_proj): Linear(in_features=768, out_features=64, bias=True)
  (topic_proj): Linear(in_features=50, out_features=64, bias=True)
  (entity_proj): Linear(in_features=100, out_features=64, bias=True)
  (context_proj): Linear(in_features=4, out_features=64, bias=True)
  (fc1): Linear(in_features=832, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=32, bias=True

In [ ]:
# ── Final Model: Step 3 — Training loop ──────────────────────────────────────
from tqdm import tqdm

final_train_losses, final_val_losses = [], []
final_val_aucs, final_val_mrrs = [], []
final_val_ndcg5s, final_val_ndcg10s = [], []
final_val_maps = []

for epoch in range(final_num_epochs):

    # ── Train ─────────────────────────────────────────────────────────────
    final_model.train()
    total_loss, total_samples = 0.0, 0

    for (user_num, user_cat, cand_news_num, cand_news_cat,
         cand_bert, cand_topic, cand_entity,
         swapped_knowledge, swapped_context,
         impression_ids_batch) in tqdm(final_train_loader,
                                       desc=f"Epoch {epoch+1}/{final_num_epochs} [train]",
                                       unit="batch"):

        user_num         = user_num.to(device)
        user_cat         = {k: v.to(device) for k, v in user_cat.items()}
        cand_news_num    = cand_news_num.to(device)
        cand_news_cat    = cand_news_cat.to(device)
        cand_bert        = cand_bert.to(device)
        cand_topic       = cand_topic.to(device)
        cand_entity      = cand_entity.to(device)
        actual_context   = swapped_context.to(device)
        actual_knowledge = swapped_knowledge.to(device)

        B = user_num.shape[0]
        final_optimizer.zero_grad()

        scores = score_candidates_ablation(
            final_model, user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            actual_context, actual_knowledge, device
        )
        loss = F.cross_entropy(scores, torch.zeros(B, dtype=torch.long, device=device))
        loss.backward()
        final_optimizer.step()

        total_loss    += loss.item() * B
        total_samples += B

    avg_train_loss = total_loss / total_samples
    final_train_losses.append(avg_train_loss)

    # ── Validate ──────────────────────────────────────────────────────────
    final_model.eval()
    val_labels_all, val_scores_all, val_imp_ids = [], [], []

    with torch.no_grad():
        for (user_num, user_cat, cand_news_num, cand_news_cat,
             cand_bert, cand_topic, cand_entity,
             context_batch, knowledge, labels_batch,
             impression_ids_batch) in final_val_loader:

            user_num      = user_num.to(device)
            user_cat      = {k: v.to(device) for k, v in user_cat.items()}
            cand_news_num = cand_news_num.to(device)
            cand_news_cat = cand_news_cat.to(device)
            cand_bert     = cand_bert.to(device)
            cand_topic    = cand_topic.to(device)
            cand_entity   = cand_entity.to(device)
            context_batch = context_batch.to(device)
            knowledge     = knowledge.to(device)

            scores = score_candidates_ablation(
                final_model, user_num, user_cat,
                cand_news_num, cand_news_cat,
                cand_bert, cand_topic, cand_entity,
                context_batch, knowledge, device
            )
            val_labels_all.extend(labels_batch.cpu().numpy().tolist())
            val_scores_all.extend(
                torch.softmax(scores, dim=1).cpu().numpy().tolist()
            )
            val_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

    flat_labels = [l for row in val_labels_all for l in row]
    flat_scores = [s for row in val_scores_all for s in row]
    val_auc = roc_auc_score(flat_labels, flat_scores) \
              if len(np.unique(flat_labels)) > 1 else 0.0
    val_mrr, val_map, val_ndcg5, val_ndcg10 = calculate_ranking_metrics(
        val_imp_ids, val_labels_all, val_scores_all
    )

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | "
          f"AUC: {val_auc:.4f}  MRR: {val_mrr:.4f}  "
          f"nDCG@5: {val_ndcg5:.4f}  nDCG@10: {val_ndcg10:.4f}")

    final_val_losses.append(val_auc)
    final_val_aucs.append(val_auc)
    final_val_mrrs.append(val_mrr)
    final_val_ndcg5s.append(val_ndcg5)
    final_val_ndcg10s.append(val_ndcg10)
    final_val_maps.append(val_map)

best_epoch_idx = final_val_ndcg5s.index(max(final_val_ndcg5s))
print("\nFinal Model Training Complete")
print(f"Best epoch: {best_epoch_idx + 1}  (by val nDCG@5)")
print(f"  nDCG@5  : {final_val_ndcg5s[best_epoch_idx]:.4f}")
print(f"  nDCG@10 : {final_val_ndcg10s[best_epoch_idx]:.4f}")
print(f"  AUC     : {final_val_aucs[best_epoch_idx]:.4f}")
print(f"  MRR     : {final_val_mrrs[best_epoch_idx]:.4f}")
print(f"  MAP     : {final_val_maps[best_epoch_idx]:.4f}")

Epoch 1/6 [train]: 100%|██████████| 4424/4424 [00:47<00:00, 92.52batch/s]


Epoch 1 | Train Loss: 1.4118 | AUC: 0.7221  MRR: 0.3240  nDCG@5: 0.3007  nDCG@10: 0.3603


Epoch 2/6 [train]: 100%|██████████| 4424/4424 [00:48<00:00, 91.71batch/s]


Epoch 2 | Train Loss: 1.3516 | AUC: 0.7299  MRR: 0.3326  nDCG@5: 0.3114  nDCG@10: 0.3721


Epoch 3/6 [train]: 100%|██████████| 4424/4424 [00:48<00:00, 91.58batch/s]


...

Epoch 4 | Train Loss: 1.3111 | AUC: 0.7348  MRR: 0.3405  nDCG@5: 0.3189  nDCG@10: 0.3793


Epoch 5/6 [train]: 100%|██████████| 4424/4424 [00:48<00:00, 92.04batch/s]


Epoch 5 | Train Loss: 1.2953 | AUC: 0.7368  MRR: 0.3462  nDCG@5: 0.3245  nDCG@10: 0.3836


Epoch 6/6 [train]: 100%|██████████| 4424/4424 [00:48<00:00, 90.90batch/s]


Epoch 6 | Train Loss: 1.2797 | AUC: 0.7358  MRR: 0.3464  nDCG@5: 0.3237  nDCG@10: 0.3838

Final Model Training Complete
Best epoch: 5  (by val nDCG@5)
  nDCG@5  : 0.3245
  nDCG@10 : 0.3836
  AUC     : 0.7368
  MRR     : 0.3462
  MAP     : 0.3194


In [ ]:
# ── Final Model: Step 4 — Test evaluation ────────────────────────────────────

final_model.eval()
test_labels_all, test_scores_all, test_imp_ids = [], [], []

with torch.no_grad():
    for (user_num, user_cat, cand_news_num, cand_news_cat,
         cand_bert, cand_topic, cand_entity,
         context_features, knowledge, labels_batch,
         impression_ids_batch) in final_test_loader:

        user_num         = user_num.to(device)
        user_cat         = {k: v.to(device) for k, v in user_cat.items()}
        cand_news_num    = cand_news_num.to(device)
        cand_news_cat    = cand_news_cat.to(device)
        cand_bert        = cand_bert.to(device)
        cand_topic       = cand_topic.to(device)
        cand_entity      = cand_entity.to(device)
        context_features = context_features.to(device)
        knowledge        = knowledge.to(device)

        scores = score_candidates_ablation(
            final_model, user_num, user_cat,
            cand_news_num, cand_news_cat,
            cand_bert, cand_topic, cand_entity,
            context_features, knowledge, device
        )
        test_labels_all.extend(labels_batch.cpu().numpy().tolist())
        test_scores_all.extend(
            torch.softmax(scores, dim=1).cpu().numpy().tolist()
        )
        test_imp_ids.extend(impression_ids_batch.cpu().numpy().tolist())

test_mrr, test_map, test_ndcg5, test_ndcg10 = calculate_ranking_metrics(
    test_imp_ids, test_labels_all, test_scores_all
)
flat_test_labels = [l for row in test_labels_all for l in row]
flat_test_scores = [s for row in test_scores_all for s in row]
test_auc = roc_auc_score(flat_test_labels, flat_test_scores) \
           if len(np.unique(flat_test_labels)) > 1 else 0.0

print("=" * 55)
print("FINAL MODEL — TEST RESULTS")
print("=" * 55)
print(f"  AUC     : {test_auc:.4f}")
print(f"  MRR     : {test_mrr:.4f}")
print(f"  MAP     : {test_map:.4f}")
print(f"  nDCG@5  : {test_ndcg5:.4f}")
print(f"  nDCG@10 : {test_ndcg10:.4f}")
print("=" * 55)
print(f"Final    nDCG@5 (test)     : {test_ndcg5:.4f}")

FINAL MODEL — TEST RESULTS
  AUC     : 0.7152
  MRR     : 0.2984
  MAP     : 0.2739
  nDCG@5  : 0.2716
  nDCG@10 : 0.3346
Final    nDCG@5 (test)     : 0.2716
